Fetch all 12 NCT registry records from ClinicalTrials.gov API v2

In [1]:

import requests, json, time, csv, os
from datetime import datetime

# ── ClinicalTrials.gov API v2 base ──────────────────────────────────────────
BASE = "https://clinicaltrials.gov/api/v2/studies"

# All NCTs in the comparator universe + BreAK CRC-001
UNIVERSE = {
    "NCT03050814": "QUILT-2.004",
    "NCT05141721": "GRANITE GO-010",
    "NCT03414983": "CheckMate 9X8",
    "NCT05608044": "BOT/BAL Phase 2",
    "NCT03860272": "BOT/BAL Phase 1",
    "NCT04194359": "Sintilimab+bev+XELOX",
    "NCT05970302": "Tislelizumab+XELOX+bev",
    "NCT06566755": "Cadonilimab+CapeOX+bev",
    "NCT03639714": "GRANITE GO-004",
    "NCT07500298": "SAR445877+mFOLFOX6+bev",
    "NCT06149481": "STAR15",
    "NCT06934538": "BreAK CRC-001 (STC-1010)",
    # Subgroup-rich studies (no NCT — will retrieve via search)
    # AtezoTRIBE, CO.26, REGONIVO, RIN handled separately
}

FIELDS = (
    "protocolSection.identificationModule,"
    "protocolSection.statusModule,"
    "protocolSection.sponsorCollaboratorsModule,"
    "protocolSection.descriptionModule,"
    "protocolSection.conditionsModule,"
    "protocolSection.designModule,"
    "protocolSection.armsInterventionsModule,"
    "protocolSection.eligibilityModule,"
    "protocolSection.outcomesModule,"
    "resultsSection.baselineCharacteristicsModule,"
    "resultsSection.outcomeMeasuresModule,"
    "resultsSection.adverseEventsModule,"
    "derivedSection.conditionBrowseModule"
)

records = {}
errors = []

for nct, alias in UNIVERSE.items():
    url = f"{BASE}/{nct}?fields={FIELDS}"
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            records[nct] = r.json()
            print(f"  ✓ {nct} ({alias})")
        else:
            errors.append((nct, alias, r.status_code))
            print(f"  ✗ {nct} ({alias}) — HTTP {r.status_code}")
    except Exception as e:
        errors.append((nct, alias, str(e)))
        print(f"  ✗ {nct} ({alias}) — {e}")
    time.sleep(0.3)

print(f"\nRetrieved: {len(records)}/{len(UNIVERSE)}  |  Errors: {len(errors)}")


  ✓ NCT03050814 (QUILT-2.004)
  ✓ NCT05141721 (GRANITE GO-010)
  ✓ NCT03414983 (CheckMate 9X8)
  ✓ NCT05608044 (BOT/BAL Phase 2)
  ✓ NCT03860272 (BOT/BAL Phase 1)
  ✓ NCT04194359 (Sintilimab+bev+XELOX)
  ✓ NCT05970302 (Tislelizumab+XELOX+bev)
  ✓ NCT06566755 (Cadonilimab+CapeOX+bev)
  ✓ NCT03639714 (GRANITE GO-004)
  ✓ NCT07500298 (SAR445877+mFOLFOX6+bev)
  ✓ NCT06149481 (STAR15)
  ✓ NCT06934538 (BreAK CRC-001 (STC-1010))

Retrieved: 12/12  |  Errors: 0


Extract structured fields from all 12 registry records

In [3]:

import json, re

def safe(d, *keys, default="NOT FOUND"):
    """Safe nested dict access."""
    for k in keys:
        if isinstance(d, dict):
            d = d.get(k, {})
        elif isinstance(d, list) and isinstance(k, int):
            d = d[k] if k < len(d) else {}
        else:
            return default
    return d if d not in ({}, [], None, "") else default

def join_list(lst, sep=" | "):
    if isinstance(lst, list):
        return sep.join(str(x) for x in lst if x)
    return str(lst) if lst else "NOT FOUND"

rows = []

for nct, alias in UNIVERSE.items():
    rec = records.get(nct, {})
    ps  = rec.get("protocolSection", {})
    rs  = rec.get("resultsSection", {})

    id_mod     = ps.get("identificationModule", {})
    stat_mod   = ps.get("statusModule", {})
    spon_mod   = ps.get("sponsorCollaboratorsModule", {})
    des_mod    = ps.get("descriptionModule", {})
    cond_mod   = ps.get("conditionsModule", {})
    design_mod = ps.get("designModule", {})
    arms_mod   = ps.get("armsInterventionsModule", {})
    elig_mod   = ps.get("eligibilityModule", {})
    out_mod    = ps.get("outcomesModule", {})

    # ── Identification ──────────────────────────────────────────────────────
    brief_title  = safe(id_mod, "briefTitle")
    official_title = safe(id_mod, "officialTitle")
    org_study_id = safe(id_mod, "orgStudyIdInfo", "id")
    secondary_ids = join_list([x.get("id","") for x in id_mod.get("secondaryIdInfos", [])])

    # ── Status ───────────────────────────────────────────────────────────────
    overall_status   = safe(stat_mod, "overallStatus")
    start_date       = safe(stat_mod, "startDateStruct", "date")
    primary_comp     = safe(stat_mod, "primaryCompletionDateStruct", "date")
    study_comp       = safe(stat_mod, "completionDateStruct", "date")
    last_update      = safe(stat_mod, "lastUpdateSubmitDate")
    enrollment_count = safe(design_mod, "enrollmentInfo", "count")
    enrollment_type  = safe(design_mod, "enrollmentInfo", "type")
    why_stopped      = safe(stat_mod, "whyStopped")

    # ── Sponsor ──────────────────────────────────────────────────────────────
    lead_sponsor = safe(spon_mod, "leadSponsor", "name")
    sponsor_class = safe(spon_mod, "leadSponsor", "class")
    collaborators = join_list([x.get("name","") for x in spon_mod.get("collaborators", [])])

    # ── Conditions ───────────────────────────────────────────────────────────
    conditions = join_list(cond_mod.get("conditions", []))
    keywords   = join_list(cond_mod.get("keywords", []))

    # ── Design ───────────────────────────────────────────────────────────────
    phases        = join_list(design_mod.get("phases", []))
    study_type    = safe(design_mod, "studyType")
    allocation    = safe(design_mod, "designInfo", "allocation")
    intervention_model = safe(design_mod, "designInfo", "interventionModel")
    masking       = safe(design_mod, "designInfo", "maskingInfo", "masking")
    primary_purpose = safe(design_mod, "designInfo", "primaryPurpose")

    # ── Arms ─────────────────────────────────────────────────────────────────
    arms = arms_mod.get("armGroups", [])
    arm_labels = join_list([f"{a.get('label','')} [{a.get('type','')}]" for a in arms])
    n_arms = len(arms)

    interventions = arms_mod.get("interventions", [])
    intervention_names = join_list([f"{i.get('type','')}: {i.get('name','')}" for i in interventions])

    # ── Eligibility ──────────────────────────────────────────────────────────
    min_age    = safe(elig_mod, "minimumAge")
    max_age    = safe(elig_mod, "maximumAge")
    sex        = safe(elig_mod, "sex")
    healthy_vol = safe(elig_mod, "healthyVolunteers")
    criteria_raw = safe(elig_mod, "eligibilityCriteria")
    # Truncate for index — full text stored separately
    criteria_short = criteria_raw[:500] + "..." if len(str(criteria_raw)) > 500 else criteria_raw

    # ── Outcomes ─────────────────────────────────────────────────────────────
    primary_outcomes = out_mod.get("primaryOutcomes", [])
    primary_out_str  = join_list([f"{o.get('measure','')} [{o.get('timeFrame','')}]" for o in primary_outcomes])
    secondary_outcomes = out_mod.get("secondaryOutcomes", [])
    secondary_out_str  = join_list([f"{o.get('measure','')} [{o.get('timeFrame','')}]" for o in secondary_outcomes[:5]])

    # ── Results summary ──────────────────────────────────────────────────────
    has_results = "YES" if rs else "NO"

    rows.append({
        "nct": nct,
        "alias": alias,
        "brief_title": brief_title,
        "official_title": official_title,
        "org_study_id": org_study_id,
        "secondary_ids": secondary_ids,
        "overall_status": overall_status,
        "why_stopped": why_stopped,
        "phases": phases,
        "study_type": study_type,
        "allocation": allocation,
        "intervention_model": intervention_model,
        "masking": masking,
        "primary_purpose": primary_purpose,
        "n_arms": n_arms,
        "arm_labels": arm_labels,
        "intervention_names": intervention_names,
        "enrollment_count": enrollment_count,
        "enrollment_type": enrollment_type,
        "lead_sponsor": lead_sponsor,
        "sponsor_class": sponsor_class,
        "collaborators": collaborators,
        "conditions": conditions,
        "keywords": keywords,
        "min_age": min_age,
        "max_age": max_age,
        "sex": sex,
        "healthy_volunteers": healthy_vol,
        "eligibility_criteria_short": criteria_short,
        "primary_outcomes": primary_out_str,
        "secondary_outcomes_top5": secondary_out_str,
        "start_date": start_date,
        "primary_completion_date": primary_comp,
        "study_completion_date": study_comp,
        "last_update": last_update,
        "has_results_section": has_results,
    })

print(f"Extracted {len(rows)} trial records")
for r in rows:
    print(f"  {r['nct']} | {r['alias'][:35]:<35} | status={r['overall_status']:<25} | phase={r['phases']:<10} | n={r['enrollment_count']:<6} | results={r['has_results_section']}")


Extracted 12 trial records
  NCT03050814 | QUILT-2.004                         | status=TERMINATED                | phase=PHASE2     | n=30     | results=YES
  NCT05141721 | GRANITE GO-010                      | status=ACTIVE_NOT_RECRUITING     | phase=PHASE2 | PHASE3 | n=700    | results=NO
  NCT03414983 | CheckMate 9X8                       | status=COMPLETED                 | phase=PHASE2 | PHASE3 | n=196    | results=YES
  NCT05608044 | BOT/BAL Phase 2                     | status=ACTIVE_NOT_RECRUITING     | phase=PHASE2     | n=234    | results=NO
  NCT03860272 | BOT/BAL Phase 1                     | status=ACTIVE_NOT_RECRUITING     | phase=PHASE1     | n=499    | results=NO
  NCT04194359 | Sintilimab+bev+XELOX                | status=UNKNOWN                   | phase=PHASE2     | n=25     | results=NO
  NCT05970302 | Tislelizumab+XELOX+bev              | status=RECRUITING                | phase=PHASE2     | n=52     | results=NO
  NCT06566755 | Cadonilimab+CapeOX+bev             

Save master trial index CSV and spot-check key fields

In [5]:

import pandas as pd, os

# Save master trial index
os.makedirs("/workspace/phd_outputs", exist_ok=True)
df = pd.DataFrame(rows)
df.to_csv("/workspace/phd_outputs/master_trial_index.csv", index=False)
print(f"Saved master_trial_index.csv — {len(df)} rows x {len(df.columns)} columns")

# Quick sanity check on key fields
print("\n── Key field spot-check ──")
for _, row in df.iterrows():
    print(f"\n{row['nct']} | {row['alias']}")
    print(f"  allocation:    {row['allocation']}")
    print(f"  n_arms:        {row['n_arms']}")
    print(f"  enrollment:    {row['enrollment_count']} ({row['enrollment_type']})")
    print(f"  why_stopped:   {row['why_stopped']}")
    print(f"  primary_out:   {str(row['primary_outcomes'])[:120]}")


Saved master_trial_index.csv — 12 rows x 36 columns

── Key field spot-check ──

NCT03050814 | QUILT-2.004
  allocation:    RANDOMIZED
  n_arms:        2
  enrollment:    30 (ACTUAL)
  why_stopped:   Terminated after an unplanned interim efficacy analysis.
  primary_out:   Progression Free Survival Between the Standard of Care (SOC) Arm A, and Standard of Care (SOC) Arm B + lead-in [Up to 1.

NCT05141721 | GRANITE GO-010
  allocation:    RANDOMIZED
  n_arms:        2
  enrollment:    700 (ESTIMATED)
  why_stopped:   NOT FOUND
  primary_out:   Phase 2: Molecular response defined as ≥ 30% decrease from baseline in circulating tumor DNA (ctDNA) [Baseline and up to

NCT03414983 | CheckMate 9X8
  allocation:    RANDOMIZED
  n_arms:        2
  enrollment:    196 (ACTUAL)
  why_stopped:   NOT FOUND
  primary_out:   Progression Free Survival (PFS) Per Blinded Independent Central Review (BICR) [From randomization to up to the date of t

NCT05608044 | BOT/BAL Phase 2
  allocation:    RANDOMIZED


Deep extract Priority 1 trial fields: eligibility, outcomes, arms, results

In [7]:

# ── Deep extraction: Priority 1 trials ──────────────────────────────────────
# QUILT-2.004, GRANITE GO-010, CheckMate 9X8, BOT/BAL Phase 2

P1_NCTS = ["NCT03050814", "NCT05141721", "NCT03414983", "NCT05608044"]

evidence_packets = {}

for nct in P1_NCTS:
    rec = records[nct]
    ps  = rec.get("protocolSection", {})
    rs  = rec.get("resultsSection", {})
    alias = UNIVERSE[nct]

    elig_mod   = ps.get("eligibilityModule", {})
    out_mod    = ps.get("outcomesModule", {})
    arms_mod   = ps.get("armsInterventionsModule", {})
    design_mod = ps.get("designModule", {})
    stat_mod   = ps.get("statusModule", {})
    id_mod     = ps.get("identificationModule", {})

    # Full eligibility criteria
    full_criteria = safe(elig_mod, "eligibilityCriteria")

    # All outcomes
    all_primary   = out_mod.get("primaryOutcomes", [])
    all_secondary = out_mod.get("secondaryOutcomes", [])
    all_other     = out_mod.get("otherOutcomes", [])

    # Arms detail
    arms = arms_mod.get("armGroups", [])
    interventions = arms_mod.get("interventions", [])

    # Results section
    outcome_measures = []
    if rs:
        for om in rs.get("outcomeMeasuresModule", {}).get("outcomeMeasures", []):
            title = om.get("title", "")
            om_type = om.get("type", "")
            classes = om.get("classes", [])
            for cls in classes:
                for cat in cls.get("categories", []):
                    for meas in cat.get("measurements", []):
                        outcome_measures.append({
                            "outcome_title": title,
                            "outcome_type": om_type,
                            "group_id": meas.get("groupId",""),
                            "value": meas.get("value",""),
                            "lower_limit": meas.get("lowerLimit",""),
                            "upper_limit": meas.get("upperLimit",""),
                        })

    # Baseline characteristics
    baseline_groups = []
    if rs:
        bg_mod = rs.get("baselineCharacteristicsModule", {})
        for grp in bg_mod.get("groups", []):
            baseline_groups.append({
                "id": grp.get("id",""),
                "title": grp.get("title",""),
                "description": grp.get("description","")[:300],
            })

    evidence_packets[nct] = {
        "alias": alias,
        "nct": nct,
        "full_eligibility_criteria": full_criteria,
        "primary_outcomes": all_primary,
        "secondary_outcomes": all_secondary,
        "other_outcomes": all_other,
        "arms": arms,
        "interventions": interventions,
        "outcome_measures_raw": outcome_measures,
        "baseline_groups": baseline_groups,
        "has_results": bool(rs),
    }

    print(f"\n{'='*70}")
    print(f"  {nct} | {alias}")
    print(f"  Primary outcomes: {len(all_primary)}")
    print(f"  Secondary outcomes: {len(all_secondary)}")
    print(f"  Arms: {len(arms)}")
    print(f"  Interventions: {len(interventions)}")
    print(f"  Results outcome measures: {len(outcome_measures)}")
    print(f"  Baseline groups: {len(baseline_groups)}")
    print(f"\n  ── ELIGIBILITY (first 1200 chars) ──")
    print(f"  {str(full_criteria)[:1200]}")



  NCT03050814 | QUILT-2.004
  Primary outcomes: 1
  Secondary outcomes: 1
  Arms: 2
  Interventions: 8
  Results outcome measures: 6
  Baseline groups: 4

  ── ELIGIBILITY (first 1200 chars) ──
  * INCLUSION CRITERIA:
* Subjects must have previously untreated metastatic or unresectable colorectal cancer and have no contraindications to treatment with the standard of care regimen as determined by the investigator. Prior adjuvant therapy for surgically resectable disease (including oligometastatic disease) is acceptable (including immunotherapy) but must have been completed at least 6 months prior to enrollment.
* Patients should not be eligible for potentially curative surgical intervention in the case of oligometastatic disease at the time of enrollment or must have actively refused after explicit discussion of potential benefit of this intervention with multidisciplinary team.
* Histologically confirmed colorectal cancer
* Patients must have measurable disease by Response Evaluation 

Extract results section data for QUILT-2.004 and CheckMate 9X8

In [9]:

# ── Extract results data for QUILT-2.004 and CheckMate 9X8 (have results sections) ──

for nct in ["NCT03050814", "NCT03414983"]:
    rec = records[nct]
    rs  = rec.get("resultsSection", {})
    alias = UNIVERSE[nct]

    print(f"\n{'='*70}")
    print(f"RESULTS SECTION: {nct} | {alias}")

    # Participant flow
    pf = rs.get("participantFlowModule", {})
    groups = pf.get("groups", [])
    print(f"\n  Participant flow groups:")
    for g in groups:
        print(f"    [{g.get('id','')}] {g.get('title','')} — {g.get('description','')[:200]}")

    # Outcome measures — focus on primary
    om_mod = rs.get("outcomeMeasuresModule", {})
    for om in om_mod.get("outcomeMeasures", []):
        if om.get("type","") == "PRIMARY":
            print(f"\n  PRIMARY OUTCOME: {om.get('title','')}")
            print(f"  Time frame: {om.get('timeFrame','')}")
            print(f"  Description: {str(om.get('description',''))[:300]}")
            # Groups
            for grp in om.get("groups", []):
                print(f"    Group [{grp.get('id','')}]: {grp.get('title','')} — n={grp.get('numSubjects','?')}")
            # Classes/categories/measurements
            for cls in om.get("classes", []):
                for cat in cls.get("categories", []):
                    for meas in cat.get("measurements", []):
                        print(f"    Measurement: groupId={meas.get('groupId','')} value={meas.get('value','')} "
                              f"lowerLimit={meas.get('lowerLimit','')} upperLimit={meas.get('upperLimit','')} "
                              f"spread={meas.get('spread','')}")

    # Adverse events summary
    ae_mod = rs.get("adverseEventsModule", {})
    print(f"\n  Adverse events time frame: {ae_mod.get('timeFrame','NOT FOUND')}")
    for grp in ae_mod.get("eventGroups", []):
        print(f"    AE group [{grp.get('id','')}]: {grp.get('title','')} "
              f"n={grp.get('subjectsAffected','?')}/{grp.get('subjectsAtRisk','?')}")



RESULTS SECTION: NCT03050814 | QUILT-2.004

  Participant flow groups:

  PRIMARY OUTCOME: Progression Free Survival Between the Standard of Care (SOC) Arm A, and Standard of Care (SOC) Arm B + lead-in
  Time frame: Up to 1.5 years
  Description: Progression free survival is defined as the duration of time from start of treatment to time of progression or death, whichever occurs first. Progression was measured by the Response Evaluation Criteria in Solid Tumors (RECIST) and is defined as at least a 20% increase in the sum of the diameters of
    Group [OG000]: Standard of Care (SOC) - Arm A — n=?
    Group [OG001]: Standard of Care (SOC) - Arm B + Lead in — n=?
    Measurement: groupId=OG000 value=8.8 lowerLimit=3.3 upperLimit=17.0 spread=
    Measurement: groupId=OG001 value=10.1 lowerLimit=3.6 upperLimit=16.1 spread=

  Adverse events time frame: Date treatment consent signed to date off study, approximately 33 months and 20 days for Arm A and 51 months and 7 days for Arm B.
    AE 

List all 70 outcome measures from CheckMate 9X8 results section

In [11]:

# ── Extract ALL outcome measures from CheckMate 9X8 (70 measures) ──
# Focus on PFS HR, OS HR, ORR, subgroup data

nct = "NCT03414983"
rec = records[nct]
rs  = rec.get("resultsSection", {})
om_mod = rs.get("outcomeMeasuresModule", {})

print("CheckMate 9X8 — All outcome measure titles:")
for i, om in enumerate(om_mod.get("outcomeMeasures", [])):
    print(f"  [{i:02d}] [{om.get('type','')}] {om.get('title','')[:100]}")


CheckMate 9X8 — All outcome measure titles:
  [00] [PRIMARY] Progression Free Survival (PFS) Per Blinded Independent Central Review (BICR)
  [01] [SECONDARY] Progression Free Survival (PFS) Per Investigator Assessment
  [02] [SECONDARY] Objective Response Rate (ORR) Per Blinded Independent Central Review (BICR)
  [03] [SECONDARY] Objective Response Rate (ORR) Per Investigator Assessment
  [04] [SECONDARY] Duration of Response (DoR) Per Blinded Independent Central Review (BICR)
  [05] [SECONDARY] Duration of Response (DoR) Per Investigator Assessment
  [06] [SECONDARY] Time to Objective Response Per Blinded Independent Central Review (BICR)
  [07] [SECONDARY] Time to Objective Response Per Investigator Assessment
  [08] [SECONDARY] Overall Survival (OS)
  [09] [SECONDARY] Number of Participants With Adverse Events (AEs)
  [10] [SECONDARY] Number of Participants With Serious Adverse Events (SAEs)
  [11] [SECONDARY] Number of Participants Experiencing Death
  [12] [SECONDARY] Number of Pa

Extract PFS, ORR, OS, DCR numeric results from CheckMate 9X8

In [13]:

# ── Extract key numeric results from CheckMate 9X8 ──
# PFS (primary), OS, ORR, DCR — with group labels and CIs

nct = "NCT03414983"
rec = records[nct]
rs  = rec.get("resultsSection", {})
om_mod = rs.get("outcomeMeasuresModule", {})

# Build group ID → title map
group_map = {}
for om in om_mod.get("outcomeMeasures", []):
    for grp in om.get("groups", []):
        group_map[grp.get("id","")] = grp.get("title","")

print("Group map:", group_map)

# Extract key outcomes
KEY_INDICES = [0, 2, 8, 14]  # PFS, ORR, OS, DCR
for idx in KEY_INDICES:
    om = om_mod.get("outcomeMeasures", [])[idx]
    print(f"\n── {om.get('type','')} | {om.get('title','')}")
    print(f"   Time frame: {om.get('timeFrame','')}")
    for cls in om.get("classes", []):
        cls_title = cls.get("title","(main)")
        for cat in cls.get("categories", []):
            cat_title = cat.get("title","(main)")
            for meas in cat.get("measurements", []):
                gid = meas.get("groupId","")
                gname = group_map.get(gid, gid)
                val = meas.get("value","")
                lo  = meas.get("lowerLimit","")
                hi  = meas.get("upperLimit","")
                spread = meas.get("spread","")
                print(f"   [{gname}] val={val} CI=({lo},{hi}) spread={spread}")


Group map: {'OG000': 'Arm A: NIV+mFOLFOX+BEV', 'OG001': 'Arm B: mFOLFOX+BEV'}

── PRIMARY | Progression Free Survival (PFS) Per Blinded Independent Central Review (BICR)
   Time frame: From randomization to up to the date of the first documented progression (up to 16 months)
   [Arm A: NIV+mFOLFOX+BEV] val=11.86 CI=(8.94,15.70) spread=
   [Arm B: mFOLFOX+BEV] val=11.93 CI=(10.09,12.19) spread=

── SECONDARY | Objective Response Rate (ORR) Per Blinded Independent Central Review (BICR)
   Time frame: From the date of randomization up to the date of objectively documented progression or the date of subsequent anticancer therapy, whichever occurs first (up to approximately 44 months)
   [Arm A: NIV+mFOLFOX+BEV] val=60.6 CI=(51.6,69.2) spread=
   [Arm B: mFOLFOX+BEV] val=45.6 CI=(33.5,58.1) spread=

── SECONDARY | Overall Survival (OS)
   Time frame: From the date of randomization up to the date of death (up to 44 months)
   [Arm A: NIV+mFOLFOX+BEV] val=30.52 CI=(25.20,39.39) spread=
   [Ar

Extract all QUILT-2.004 results and baseline characteristics

In [15]:

# ── Extract QUILT-2.004 results in detail ──
nct = "NCT03050814"
rec = records[nct]
rs  = rec.get("resultsSection", {})
om_mod = rs.get("outcomeMeasuresModule", {})

print("QUILT-2.004 — All outcome measures:")
for i, om in enumerate(om_mod.get("outcomeMeasures", [])):
    print(f"  [{i:02d}] [{om.get('type','')}] {om.get('title','')[:100]}")

# Group map
group_map_q = {}
for om in om_mod.get("outcomeMeasures", []):
    for grp in om.get("groups", []):
        group_map_q[grp.get("id","")] = grp.get("title","")
print(f"\nGroup map: {group_map_q}")

# All outcomes with values
for om in om_mod.get("outcomeMeasures", []):
    print(f"\n── {om.get('type','')} | {om.get('title','')}")
    print(f"   Time frame: {om.get('timeFrame','')}")
    for cls in om.get("classes", []):
        for cat in cls.get("categories", []):
            for meas in cat.get("measurements", []):
                gid = meas.get("groupId","")
                gname = group_map_q.get(gid, gid)
                val = meas.get("value","")
                lo  = meas.get("lowerLimit","")
                hi  = meas.get("upperLimit","")
                print(f"   [{gname}] val={val} CI=({lo},{hi})")

# Baseline characteristics
bg_mod = rs.get("baselineCharacteristicsModule", {})
print(f"\n── BASELINE CHARACTERISTICS ──")
print(f"  Groups: {[g.get('title','') for g in bg_mod.get('groups',[])]}")
for meas in bg_mod.get("measures", []):
    print(f"  Measure: {meas.get('title','')[:80]}")
    for cls in meas.get("classes", []):
        for cat in cls.get("categories", []):
            vals = [(m.get("groupId",""), m.get("value","")) for m in cat.get("measurements",[])]
            print(f"    {cat.get('title','(main)')}: {vals}")


QUILT-2.004 — All outcome measures:
  [00] [PRIMARY] Progression Free Survival Between the Standard of Care (SOC) Arm A, and Standard of Care (SOC) Arm B
  [01] [SECONDARY] Number of Participants That Are Hospitalized Because of Adverse Events Attributed to Disease Progres
  [02] [OTHER_PRE_SPECIFIED] Number of Participants With Serious and Non-serious Adverse Events Assessed by the Common Terminolog

Group map: {'OG000': 'Standard of Care (SOC) - Arm A', 'OG001': 'Standard of Care (SOC) - Arm B + Lead in'}

── PRIMARY | Progression Free Survival Between the Standard of Care (SOC) Arm A, and Standard of Care (SOC) Arm B + lead-in
   Time frame: Up to 1.5 years
   [Standard of Care (SOC) - Arm A] val=8.8 CI=(3.3,17.0)
   [Standard of Care (SOC) - Arm B + Lead in] val=10.1 CI=(3.6,16.1)

── SECONDARY | Number of Participants That Are Hospitalized Because of Adverse Events Attributed to Disease Progression
   Time frame: Up to 51.5 months
   [Standard of Care (SOC) - Arm A] val=4 CI=(,)
 

PubMed search and abstract retrieval for Priority 1 trials

In [17]:

# ── PubMed retrieval for Priority 1 trials ──────────────────────────────────
import requests, time, xml.etree.ElementTree as ET

PUBMED_SEARCH = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
PUBMED_FETCH  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

def pubmed_search(query, retmax=5):
    r = requests.get(PUBMED_SEARCH, params={
        "db": "pubmed", "term": query, "retmax": retmax,
        "retmode": "json", "sort": "relevance"
    }, timeout=20)
    return r.json().get("esearchresult", {}).get("idlist", [])

def pubmed_abstract(pmid):
    r = requests.get(PUBMED_FETCH, params={
        "db": "pubmed", "id": pmid, "retmode": "xml", "rettype": "abstract"
    }, timeout=20)
    root = ET.fromstring(r.text)
    # Title
    title_el = root.find(".//ArticleTitle")
    title = "".join(title_el.itertext()) if title_el is not None else ""
    # Abstract
    abs_texts = []
    for ab in root.findall(".//AbstractText"):
        label = ab.get("Label","")
        text  = "".join(ab.itertext())
        abs_texts.append(f"[{label}] {text}" if label else text)
    abstract = " ".join(abs_texts)
    # Journal + year
    journal_el = root.find(".//Journal/Title")
    journal = journal_el.text if journal_el is not None else ""
    year_el = root.find(".//PubDate/Year")
    year = year_el.text if year_el is not None else ""
    # Authors
    authors = []
    for au in root.findall(".//Author")[:3]:
        ln = au.find("LastName")
        fn = au.find("ForeName")
        if ln is not None:
            authors.append(f"{ln.text} {fn.text[0] if fn is not None else ''}")
    return {
        "pmid": pmid,
        "title": title,
        "abstract": abstract[:2000],
        "journal": journal,
        "year": year,
        "authors": ", ".join(authors) + (" et al." if len(authors) >= 3 else ""),
    }

# Search queries for each Priority 1 trial
QUERIES = {
    "NCT03050814": 'NCT03050814 OR "QUILT-2.004" OR "AdCEA avelumab colorectal"',
    "NCT05141721": 'NCT05141721 OR "GRANITE GO-010" OR "GRT-C901 GRT-R902 colorectal"',
    "NCT03414983": 'NCT03414983 OR "CheckMate 9X8" OR "nivolumab mFOLFOX6 bevacizumab colorectal"',
    "NCT05608044": 'NCT05608044 OR "botensilimab balstilimab colorectal" OR "AGEN1181 AGEN2034 colorectal"',
    "NCT03860272": '"botensilimab balstilimab" OR "AGEN1181 AGEN2034" phase 1 colorectal',
}

pub_results = {}
for nct, query in QUERIES.items():
    pmids = pubmed_search(query, retmax=3)
    pubs = []
    for pmid in pmids:
        try:
            pub = pubmed_abstract(pmid)
            pubs.append(pub)
            print(f"  {nct} | PMID {pmid} | {pub['year']} | {pub['journal'][:50]}")
            print(f"    {pub['title'][:100]}")
        except Exception as e:
            print(f"  {nct} | PMID {pmid} | ERROR: {e}")
        time.sleep(0.4)
    pub_results[nct] = pubs
    print()


  NCT03050814 | PMID 35274710 | 2022 | The oncologist
    A Randomized Phase II Trial of mFOLFOX6 + Bevacizumab Alone or with AdCEA Vaccine + Avelumab Immunot

  NCT05141721 | PMID 35710971 | ERROR: not well-formed (invalid token): line 1, column 0
  NCT05141721 | PMID 34051219 | 2021 | Toxicon : official journal of the International So
    Impact of granite irradiation on aflatoxin reduction in pistachio.
  NCT05141721 | PMID 27517327 | 2016 | Journal of radiological protection : official jour
    Assessment of occupational exposure in a granite quarry and processing factory.

  NCT03414983 | PMID 38485190 | 2024 | Journal for immunotherapy of cancer
    Modified FOLFOX6 plus bevacizumab with and without nivolumab for first-line treatment of metastatic 
  NCT03414983 | PMID 33812497 | 2021 | The Lancet. Oncology
    Health-related quality of life in patients with microsatellite instability-high or mismatch repair d

  NCT05608044 | PMID 38871975 | 2024 | Nature medicine
    Botensilim

Fix GRANITE GO-010 PubMed query

In [19]:

# Fix GRANITE GO-010 query — use Gritstone bio + neoantigen + colorectal
# Also fetch full abstracts for confirmed good PMIDs

FIXED_QUERIES = {
    "NCT05141721_GRANITE": 'Gritstone "neoantigen" "colorectal" OR "GRT-C901" OR "GRT-R902" OR "GRANITE" "mismatch repair" colorectal',
    "NCT05141721_GRANITE_v2": '"Gritstone" "colorectal" "neoantigen vaccine"',
}

for label, query in FIXED_QUERIES.items():
    pmids = pubmed_search(query, retmax=5)
    print(f"\n{label}: {pmids}")
    for pmid in pmids:
        try:
            pub = pubmed_abstract(pmid)
            print(f"  PMID {pmid} | {pub['year']} | {pub['journal'][:60]}")
            print(f"    {pub['title'][:120]}")
        except Exception as e:
            print(f"  PMID {pmid} | ERROR: {e}")
        time.sleep(0.3)



NCT05141721_GRANITE: []

NCT05141721_GRANITE_v2: ['35970920']
  PMID 35970920 | 2022 | Nature medicine
    Individualized, heterologous chimpanzee adenovirus and self-amplifying mRNA neoantigen vaccine for advanced metastatic s


Fetch full abstracts for all confirmed Priority 1 publications

In [21]:

# Fetch full abstracts for all confirmed good PMIDs
CONFIRMED_PMIDS = {
    "NCT03050814_QUILT": "35274710",
    "NCT03414983_CM9X8": "38485190",
    "NCT05608044_BOTBAL": "38871975",
    "NCT05141721_GRANITE_mech": "35970920",  # Gritstone neoantigen vaccine mechanism paper
}

full_abstracts = {}
for label, pmid in CONFIRMED_PMIDS.items():
    pub = pubmed_abstract(pmid)
    full_abstracts[label] = pub
    print(f"\n{'='*70}")
    print(f"LABEL: {label} | PMID: {pmid}")
    print(f"Title: {pub['title']}")
    print(f"Authors: {pub['authors']}")
    print(f"Journal: {pub['journal']} ({pub['year']})")
    print(f"\nAbstract:\n{pub['abstract']}")
    time.sleep(0.3)



LABEL: NCT03050814_QUILT | PMID: 35274710
Title: A Randomized Phase II Trial of mFOLFOX6 + Bevacizumab Alone or with AdCEA Vaccine + Avelumab Immunotherapy for Untreated Metastatic Colorectal Cancer.
Authors: Redman J, Tsai Y, Weinberg B et al.
Journal: The oncologist (2022)

Abstract:
[BACKGROUND] FOLFOX plus bevacizumab is a standard of care (SOC) for first-line treatment of microsatellite-stable metastatic colorectal cancer (MSS mCRC). This study randomized patients to SOC or SOC plus avelumab (anti-PD-L1) plus CEA-targeted vaccine. [METHODS] Patients with untreated MSS mCRC enrolled to a lead-in arm assessing safety of SOC + immuno-oncology agents (IO). Next, patients were randomized to SOC or SOC + IO. The primary endpoint was progression-free survival (PFS). Multiple immune parameters were analyzed. [RESULTS] Six patients enrolled to safety lead-in, 10 randomized to SOC, and 10 to SOC + IO. There was no difference in median PFS comparing SOC versus SOC + IO (8.8 months (95% CI: 

Deep extract Priority 2 trial eligibility, arms, interventions, outcomes

In [23]:

# ── Priority 2 trials: registry deep extraction ──────────────────────────────
P2_NCTS = ["NCT04194359", "NCT05970302", "NCT06566755", "NCT03639714", 
           "NCT07500298", "NCT06149481"]

p2_packets = {}
for nct in P2_NCTS:
    rec = records[nct]
    ps  = rec.get("protocolSection", {})
    alias = UNIVERSE[nct]
    elig_mod   = ps.get("eligibilityModule", {})
    out_mod    = ps.get("outcomesModule", {})
    arms_mod   = ps.get("armsInterventionsModule", {})
    design_mod = ps.get("designModule", {})

    full_criteria = safe(elig_mod, "eligibilityCriteria")
    arms = arms_mod.get("armGroups", [])
    interventions = arms_mod.get("interventions", [])
    primary_outcomes = out_mod.get("primaryOutcomes", [])

    p2_packets[nct] = {
        "alias": alias,
        "full_eligibility_criteria": full_criteria,
        "arms": arms,
        "interventions": interventions,
        "primary_outcomes": primary_outcomes,
    }

    print(f"\n{'='*60}")
    print(f"{nct} | {alias}")
    print(f"\n  ELIGIBILITY (first 1500 chars):")
    print(f"  {str(full_criteria)[:1500]}")
    print(f"\n  ARMS:")
    for a in arms:
        print(f"    [{a.get('type','')}] {a.get('label','')} — {str(a.get('description',''))[:200]}")
    print(f"\n  INTERVENTIONS:")
    for i in interventions:
        print(f"    {i.get('type','')}: {i.get('name','')} — {str(i.get('description',''))[:150]}")
    print(f"\n  PRIMARY OUTCOMES:")
    for o in primary_outcomes:
        print(f"    {o.get('measure','')} [{o.get('timeFrame','')}]")



NCT04194359 | Sintilimab+bev+XELOX

  ELIGIBILITY (first 1500 chars):
  Inclusion Criteria:

1. Male or female, age ≥ 18 years old, ≤ 75 years old;
2. Metastatic colorectal adenocarcinoma confirmed by histology and unresectable metastatic colorectal cancer confirmed by a multidisciplinary team;
3. RAS gene mutation, BRAF wild-type and microsatellite stable confirmed by polymerase chain reaction using a panel of six mononucleotide repeat markers (BAT-25, BAT-26, NR-21, NR-24, NR-27, and MONO-27);
4. ECOG performance status of 0-1;
5. Life expectancy≥3 months;
6. Adequate organ and bone marrow functions: absolute neutrophil count \>1.5×109/L, hemoglobin \>8 g/dL, platelet count \>100×109/L, prothrombin time \<1.5 upper limit of normal (ULN), activated partial thromboplastin time \<1.5 ULN, bilirubin ≤1.5×ULN (could be extended to 3×ULN in case of liver metastasis), blood aspartate aminotransferase and alanine aminotransferase ≤2.5×ULN (could be extended to 5×ULN in case of liver metasta

PubMed retrieval for subgroup-rich studies: AtezoTRIBE, CO.26, REGONIVO, RIN, IMblaze370

In [25]:

# ── Subgroup-rich studies: AtezoTRIBE, CO.26, REGONIVO, RIN ─────────────────
# These are cited as evidence anchors in Agent 3/4/5 but not in the NCT universe
# Retrieve via PubMed

SUBGROUP_QUERIES = {
    "AtezoTRIBE": '"AtezoTRIBE" OR "atezolizumab FOLFOXIRI bevacizumab colorectal" immunoscore',
    "CO26_CCTG": '"CO.26" OR "CCTG CO.26" "durvalumab tremelimumab colorectal" MSS',
    "REGONIVO": '"REGONIVO" OR "regorafenib nivolumab colorectal" MSS',
    "RIN": '"RIN trial" OR "regorafenib ipilimumab nivolumab colorectal" MSS',
    "KEYNOTE177_MSS": '"KEYNOTE-177" OR "pembrolizumab colorectal" "microsatellite stable" subgroup',
    "IMblaze370": '"IMblaze370" OR "cobimetinib atezolizumab colorectal" MSS',
}

subgroup_pubs = {}
for label, query in SUBGROUP_QUERIES.items():
    pmids = pubmed_search(query, retmax=2)
    print(f"\n{label}: found {len(pmids)} PMIDs: {pmids}")
    pubs = []
    for pmid in pmids:
        try:
            pub = pubmed_abstract(pmid)
            pubs.append(pub)
            print(f"  PMID {pmid} | {pub['year']} | {pub['journal'][:50]}")
            print(f"    {pub['title'][:110]}")
        except Exception as e:
            print(f"  PMID {pmid} | ERROR: {e}")
        time.sleep(0.35)
    subgroup_pubs[label] = pubs



AtezoTRIBE: found 2 PMIDs: ['38865678', '39908699']
  PMID 38865678 | 2024 | Journal of clinical oncology : official journal of
    Upfront Fluorouracil, Leucovorin, Oxaliplatin, and Irinotecan Plus Bevacizumab With or Without Atezolizumab fo
  PMID 39908699 | 2025 | ESMO open
    Liver metastases do not predict resistance to the addition of atezolizumab to first-line FOLFOXIRI plus bevaci

CO26_CCTG: found 0 PMIDs: []

REGONIVO: found 0 PMIDs: []

RIN: found 2 PMIDs: ['36892833', '28591523']
  PMID 36892833 | 2023 | JAMA oncology
    Regorafenib, Ipilimumab, and Nivolumab for Patients With Microsatellite Stable Colorectal Cancer and Disease P
  PMID 28591523 | 2017 | The New England journal of medicine
    Completion Dissection or Observation for Sentinel-Node Metastasis in Melanoma.

KEYNOTE177_MSS: found 2 PMIDs: ['40873566', '39765187']
  PMID 40873566 | ERROR: not well-formed (invalid token): line 1, column 0
  PMID 39765187 | 2025 | ESMO open
    Efficacy of pembrolizumab in mic

Fix CO.26 and REGONIVO PubMed queries

In [27]:

# Fix CO.26 and REGONIVO queries, fetch full abstracts for confirmed hits

FIXED_SUBGROUP = {
    "CO26_v2": '"durvalumab" "tremelimumab" "colorectal" "microsatellite stable"',
    "REGONIVO_v2": '"regorafenib" "nivolumab" "colorectal" "microsatellite stable" phase',
    "REGONIVO_v3": '"REGONIVO" colorectal',
}

for label, query in FIXED_SUBGROUP.items():
    pmids = pubmed_search(query, retmax=3)
    print(f"\n{label}: {pmids}")
    for pmid in pmids:
        try:
            pub = pubmed_abstract(pmid)
            print(f"  PMID {pmid} | {pub['year']} | {pub['journal'][:55]}")
            print(f"    {pub['title'][:110]}")
        except Exception as e:
            print(f"  PMID {pmid} | ERROR: {e}")
        time.sleep(0.3)



CO26_v2: ['37563240', '38727700', '40706222']
  PMID 37563240 | 2023 | Nature medicine
    First-line durvalumab and tremelimumab with chemotherapy in RAS-mutated metastatic colorectal cancer: a phase 
  PMID 38727700 | 2024 | Clinical cancer research : an official journal of the A
    Plasma versus Tissue Tumor Mutational Burden as Biomarkers of Durvalumab plus Tremelimumab Response in Patient
  PMID 40706222 | ERROR: not well-formed (invalid token): line 1, column 0

REGONIVO_v2: ['36892833', '39504677', '32343640']
  PMID 36892833 | 2023 | JAMA oncology
    Regorafenib, Ipilimumab, and Nivolumab for Patients With Microsatellite Stable Colorectal Cancer and Disease P
  PMID 39504677 | 2024 | European journal of cancer (Oxford, England : 1990)
    Updated survival outcome of regorafenib, ipilimumab, and nivolumab in refractory microsatellite stable non-liv
  PMID 32343640 | ERROR: not well-formed (invalid token): line 1, column 0

REGONIVO_v3: ['32343640', '38336371', '37980853']
  P

Fetch full abstracts for all confirmed subgroup-rich study publications

In [29]:

# Fetch full abstracts for all confirmed subgroup-rich study PMIDs
SUBGROUP_CONFIRMED = {
    "AtezoTRIBE_main":    "38865678",
    "AtezoTRIBE_livermets": "39908699",
    "CO26_main":          "37563240",
    "CO26_biomarker":     "38727700",
    "RIN_main":           "36892833",
    "RIN_updated":        "39504677",
    "REGONIVO_main":      "32343640",
    "REGONIVO_multiomic": "38336371",
    "KEYNOTE_TMB_MSS":    "39765187",
}

subgroup_abstracts = {}
for label, pmid in SUBGROUP_CONFIRMED.items():
    try:
        pub = pubmed_abstract(pmid)
        subgroup_abstracts[label] = pub
        print(f"\n{'='*65}")
        print(f"LABEL: {label} | PMID: {pmid}")
        print(f"Title: {pub['title']}")
        print(f"Authors: {pub['authors']}")
        print(f"Journal: {pub['journal']} ({pub['year']})")
        print(f"\nAbstract:\n{pub['abstract'][:1800]}")
    except Exception as e:
        print(f"\n{label} | PMID {pmid} | ERROR: {e}")
    time.sleep(0.35)



LABEL: AtezoTRIBE_main | PMID: 38865678
Title: Upfront Fluorouracil, Leucovorin, Oxaliplatin, and Irinotecan Plus Bevacizumab With or Without Atezolizumab for Patients With Metastatic Colorectal Cancer: Updated and Overall Survival Results of the ATEZOTRIBE Study.
Authors: Antoniotti C, Rossini D, Pietrantonio F et al.
Journal: Journal of clinical oncology : official journal of the American Society of Clinical Oncology (2024)

Abstract:
Clinical trials frequently include multiple end points that mature at different times. The initial report, typically based on the primary end point, may be published when key planned co-primary or secondary analyses are not yet available. Clinical Trial Updates provide an opportunity to disseminate additional results from studies, published in JCO or elsewhere, for which the primary end point has already been reported.We report 4-year results of the phase II randomized AtezoTRIBE study. Eligible patients with metastatic colorectal cancer (mCRC) receive

Full deep extraction of BreAK CRC-001 registry record

In [31]:

# ── Step 4: BreAK CRC-001 deep extraction ────────────────────────────────────
nct = "NCT06934538"
rec = records[nct]
ps  = rec.get("protocolSection", {})

elig_mod   = ps.get("eligibilityModule", {})
out_mod    = ps.get("outcomesModule", {})
arms_mod   = ps.get("armsInterventionsModule", {})
design_mod = ps.get("designModule", {})
stat_mod   = ps.get("statusModule", {})
id_mod     = ps.get("identificationModule", {})
contacts   = ps.get("contactsLocationsModule", {})
oversight  = ps.get("oversightModule", {})
desc_mod   = ps.get("descriptionModule", {})

print("="*70)
print("BreAK CRC-001 (NCT06934538) — FULL REGISTRY EXTRACTION")
print("="*70)

print(f"\nBrief title: {safe(id_mod, 'briefTitle')}")
print(f"Official title: {safe(id_mod, 'officialTitle')}")
print(f"Org study ID: {safe(id_mod, 'orgStudyIdInfo', 'id')}")
print(f"Secondary IDs: {[x.get('id','') for x in id_mod.get('secondaryIdInfos', [])]}")
print(f"Status: {safe(stat_mod, 'overallStatus')}")
print(f"Start date: {safe(stat_mod, 'startDateStruct', 'date')}")
print(f"Primary completion: {safe(stat_mod, 'primaryCompletionDateStruct', 'date')}")
print(f"Study completion: {safe(stat_mod, 'completionDateStruct', 'date')}")
print(f"Enrollment: {safe(design_mod, 'enrollmentInfo', 'count')} ({safe(design_mod, 'enrollmentInfo', 'type')})")
print(f"Phase: {design_mod.get('phases', [])}")
print(f"Study type: {safe(design_mod, 'studyType')}")
print(f"Allocation: {safe(design_mod, 'designInfo', 'allocation')}")
print(f"Intervention model: {safe(design_mod, 'designInfo', 'interventionModel')}")
print(f"Primary purpose: {safe(design_mod, 'designInfo', 'primaryPurpose')}")

print(f"\n── DESCRIPTION ──")
print(f"Brief summary: {safe(desc_mod, 'briefSummary')[:1000]}")
print(f"\nDetailed description: {safe(desc_mod, 'detailedDescription')[:2000]}")

print(f"\n── ARMS ──")
for a in arms_mod.get("armGroups", []):
    print(f"  [{a.get('type','')}] {a.get('label','')}")
    print(f"    {str(a.get('description',''))[:400]}")

print(f"\n── INTERVENTIONS ──")
for i in arms_mod.get("interventions", []):
    print(f"  {i.get('type','')}: {i.get('name','')}")
    print(f"    {str(i.get('description',''))[:400]}")
    print(f"    Arms: {i.get('armGroupLabels', [])}")

print(f"\n── PRIMARY OUTCOMES ──")
for o in out_mod.get("primaryOutcomes", []):
    print(f"  {o.get('measure','')} [{o.get('timeFrame','')}]")
    print(f"    {str(o.get('description',''))[:300]}")

print(f"\n── SECONDARY OUTCOMES ──")
for o in out_mod.get("secondaryOutcomes", []):
    print(f"  {o.get('measure','')} [{o.get('timeFrame','')}]")

print(f"\n── OTHER/EXPLORATORY OUTCOMES ──")
for o in out_mod.get("otherOutcomes", []):
    print(f"  {o.get('measure','')} [{o.get('timeFrame','')}]")


BreAK CRC-001 (NCT06934538) — FULL REGISTRY EXTRACTION

Brief title: First-in-human Trial of STC-1010, an Immunotherapy, in Patients With Unresectable Locally Advanced or Metastatic Colorectal Cancer
Official title: Open Label, Multicenter, Dose-escalation and Cohort-expansion Phase I/IIA Trial of STC-1010, an Immunotherapy, in Patients With Unresectable Locally Advanced or Metastatic Colorectal Cancer (CRC) - BreAK CRC Trial (BreAK for Brenus Anti-cancer)
Org study ID: BreAK CRC-001
Secondary IDs: []
Status: RECRUITING
Start date: 2025-06-17
Primary completion: 2028-06
Study completion: 2029-06
Enrollment: 90 (ESTIMATED)
Phase: ['PHASE1', 'PHASE2']
Study type: INTERVENTIONAL
Allocation: NON_RANDOMIZED
Intervention model: SEQUENTIAL
Primary purpose: TREATMENT

── DESCRIPTION ──
Brief summary: This is a phase I/IIA, first-in-human (FIH), two-part, open-label, multicenter study to characterize the safety, tolerability profile, and clinical efficacy of STC-1010 associated with GM-CSF and 

Extract full eligibility criteria for BreAK CRC-001

In [33]:

# ── BreAK CRC-001: Full eligibility criteria extraction ──────────────────────
nct = "NCT06934538"
rec = records[nct]
ps  = rec.get("protocolSection", {})
elig_mod = ps.get("eligibilityModule", {})

full_criteria = safe(elig_mod, "eligibilityCriteria")
print("BreAK CRC-001 — FULL ELIGIBILITY CRITERIA:")
print(full_criteria)


BreAK CRC-001 — FULL ELIGIBILITY CRITERIA:
Inclusion Criteria:

1. Male or female aged 18-75 years
2. Histologically confirmed diagnosis of unresectable locally advanced (stage IIIC, T4b) or unresectable metastatic (stage IV) (R0) adenocarcinoma of the colon or rectum
3. Adjuvant fluoropyrimidine monotherapy or oxaliplatin-based chemotherapy allowed if more than 6 months have elapsed between the end of adjuvant treatment and first relapse
4. Determination of KRAS and BRAF mutation status
5. Measurable disease per Response Evaluation Criteria in Solid Tumors (RECIST 1.1)
6. Must agree to have biopsy at screening and on-treatment, only if not representing an unacceptable clinical risk and/or if technically feasible as judged by the Investigator in discussion with the interventional radiologist or endoscopist
7. Eastern Cooperative Oncology Group (ECOG) performance status (PS) ≤ 1. Participants \>70 years must have a PS= 0.
8. Life expectancy \> 3 months as assessed by the investigator
9.

Search PubMed for ASCO 2023 abstract e14661 and STC-1010 cell line data

In [35]:

# ── Search for ASCO 2023 abstract e14661 (STC-1010 cell line identity) ────────
# Try PubMed + web search approach

ASCO_QUERIES = [
    '"STC-1010" colorectal ASCO 2023',
    '"STC-1010" "cell line" colorectal vaccine',
    '"Brenus" "STC-1010" colorectal',
    '"haptenated" "whole cell" colorectal vaccine "cell line"',
    '"DNFB" "whole cell" colorectal vaccine clinical',
]

print("Searching PubMed for ASCO 2023 abstract e14661 / STC-1010 cell line data:")
for query in ASCO_QUERIES:
    pmids = pubmed_search(query, retmax=3)
    if pmids:
        print(f"\nQuery: {query}")
        print(f"  PMIDs: {pmids}")
        for pmid in pmids[:2]:
            try:
                pub = pubmed_abstract(pmid)
                print(f"  PMID {pmid} | {pub['year']} | {pub['journal'][:50]}")
                print(f"    {pub['title'][:100]}")
            except:
                pass
            time.sleep(0.3)
    else:
        print(f"  [0 results] {query}")


Searching PubMed for ASCO 2023 abstract e14661 / STC-1010 cell line data:
  [0 results] "STC-1010" colorectal ASCO 2023
  [0 results] "STC-1010" "cell line" colorectal vaccine
  [0 results] "Brenus" "STC-1010" colorectal
  [0 results] "haptenated" "whole cell" colorectal vaccine "cell line"
  [0 results] "DNFB" "whole cell" colorectal vaccine clinical


Build complete cross-trial biomarker/timing matrix — all 16 trials/studies

In [37]:

import pandas as pd, json, os

# ── STEP 5: Cross-trial biomarker/timing matrix ──────────────────────────────
# Structured comparison across all trials on 8 dimensions

matrix_rows = [

    # ── PRIORITY 1 ──────────────────────────────────────────────────────────
    {
        "trial": "QUILT-2.004",
        "nct": "NCT03050814",
        "comparator_class": "HARD NEGATIVE BENCHMARK",
        "evidence_quality": "LITERATURE-SUPPORTED",
        "source_pmid": "35274710",
        "line": "1L",
        "mss_restriction": "NO — all-comer mCRC (MSS not required at enrollment)",
        "liver_met_handling": "NOT RECORDED — no restriction, no stratification, no covariate",
        "liver_met_source": "Registry eligibility: no liver met criterion. Publication silent on liver met prevalence.",
        "ctdna_use": "NOT USED — no ctDNA endpoint in registry or publication",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "NOT SPECIFIED in registry. Publication reports immune parameter analysis (PBMCs, T-cell responses) but biopsy schedule not detailed.",
        "dth_immune_readouts": "YES — multifunctional CD4+/CD8+ T-cell responses to MUC1/brachyury measured by ICS. 8/11 IO patients vs 1/8 SOC (p=0.020). Qualitative (positive/negative), not quantitative DTH score.",
        "dth_platform": "Intracellular cytokine staining (ICS) for T-cell responses; cascade antigen panel",
        "dth_timepoints": "Post-treatment (timing not specified in abstract)",
        "bevacizumab_structure": "MANDATORY — mFOLFOX6 + bevacizumab in both arms",
        "bevacizumab_timing": "Standard q2w with mFOLFOX6",
        "enrichment_design": "NONE — all-comer MSS enrollment; no pre-specified biomarker enrichment",
        "enrichment_biomarker": "NOT PRE-SPECIFIED",
        "kras_subtype_capture": "NOT FOUND — KRAS mutation status not reported as subtype",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "PFS (RECIST 1.1)",
        "primary_result": "HR 1.061 (95%CI 0.380–2.966), p=0.91; mPFS 8.8 vs 10.1 months",
        "failure_mode_primary": "M — mechanism insufficient (priming without trafficking confirmed)",
        "failure_mode_contributing": "S (all-comer), A (n=30, terminated)",
        "key_biomarker_finding": "T-cell priming occurred (8/11 IO) but did not translate to PFS benefit — confirms priming-without-trafficking failure mode",
        "canon_status": "VERIFIED — HR 1.061 confirmed from registry + publication",
    },
    {
        "trial": "GRANITE GO-010",
        "nct": "NCT05141721",
        "comparator_class": "SOFT BIOMARKER-SIGNAL BENCHMARK",
        "evidence_quality": "PRESS-RELEASE-ONLY",
        "source_pmid": "NOT FOUND — no peer-reviewed publication",
        "line": "1L maintenance",
        "mss_restriction": "YES — dMMR/MSI-H excluded; BRAF V600E excluded; TMB <1 mut/Mb excluded",
        "liver_met_handling": "NOT STRATIFIED — no liver met restriction or stratification in registry",
        "liver_met_source": "Registry eligibility: no liver met criterion. ctDNA-low subgroup analysis (press release) does not report liver met breakdown.",
        "ctdna_use": "YES — PRIMARY endpoint Phase 2: ≥30% decrease from baseline ctDNA (molecular response)",
        "ctdna_platform": "Not specified in registry",
        "ctdna_timepoints": "Baseline and up to 12 weeks (per registry primary outcome timeframe)",
        "ctdna_threshold": "≥30% decrease from baseline = molecular response (Phase 2 primary); ctDNA-low subgroup threshold not defined in public sources",
        "biopsy_schedule": "REQUIRED — FFPE tumor specimens required at inclusion (registry eligibility criterion)",
        "dth_immune_readouts": "NOT REPORTED in public sources",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "MANDATORY — FOLFOX/bev, CAPEOX/bev, FOLFOXIRI/bev, or CAPOXIRI/bev (induction); FP+bev (maintenance)",
        "bevacizumab_timing": "Induction phase then maintenance; vaccine administered during maintenance",
        "enrichment_design": "POST-HOC EXPLORATORY — ctDNA-low subgroup was exploratory, not pre-specified primary analysis",
        "enrichment_biomarker": "ctDNA-low (threshold undefined in public sources)",
        "kras_subtype_capture": "NOT FOUND — KRAS mutation status not specified as subtype in registry",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "Phase 2: Molecular response (≥30% ctDNA decrease); Phase 3: PFS",
        "primary_result": "PRESS-RELEASE-ONLY: HR 0.73 all-comers; HR 0.50 ctDNA-low subgroup (both non-significant, wide CIs)",
        "failure_mode_primary": "S — patients wrong (all-comer diluted ctDNA-low signal)",
        "failure_mode_contributing": "M (personalized approach complex), A (ctDNA-low exploratory not pre-specified; company bankrupt)",
        "key_biomarker_finding": "ctDNA-low subgroup HR 0.50 is the strongest available enrichment hypothesis for vaccine IO in MSS mCRC — but press-release-only, wide CI, exploratory",
        "canon_status": "PRESS-RELEASE-ONLY — HR 0.50 and HR 0.73 must not be cited externally as evidence of efficacy",
    },
    {
        "trial": "CheckMate 9X8",
        "nct": "NCT03414983",
        "comparator_class": "NULL BENCHMARK — identical backbone",
        "evidence_quality": "PEER-REVIEWED",
        "source_pmid": "38485190",
        "line": "1L",
        "mss_restriction": "NO — all-comer mCRC (95% MSS/pMMR in enrolled population per publication)",
        "liver_met_handling": "NOT STRATIFIED — no liver met restriction or stratification. Preplanned exploratory biomarker analyses performed but liver met subgroup not reported in primary publication.",
        "liver_met_source": "Registry: no liver met criterion. Publication (PMID 38485190): liver met subgroup not reported.",
        "ctdna_use": "NOT USED as endpoint — preplanned exploratory biomarker analyses mentioned but ctDNA not specified",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "REQUIRED — 'ability to provide adequate tissue sample' in inclusion criteria",
        "dth_immune_readouts": "NOT REPORTED — no DTH or T-cell response data in publication",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "MANDATORY — nivolumab + mFOLFOX6 + bevacizumab vs mFOLFOX6 + bevacizumab",
        "bevacizumab_timing": "Standard q2w with mFOLFOX6",
        "enrichment_design": "NONE — all-comer enrollment; preplanned exploratory biomarker analyses (not pre-specified enrichment)",
        "enrichment_biomarker": "NOT PRE-SPECIFIED",
        "kras_subtype_capture": "NOT FOUND in public sources",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "PFS by BICR (RECIST 1.1)",
        "primary_result": "HR 0.81 (95%CI 0.53–1.23), p=0.30; mPFS 11.9 vs 11.9 months. ORR 60% vs 46%. OS 30.52 vs 31.77 months (HR 1.03). Phase 3 NOT triggered.",
        "failure_mode_primary": "M — PD-1 alone cannot overcome 6-layer MSS TME suppression",
        "failure_mode_contributing": "S (no liver met stratification), A (no biomarker enrichment)",
        "key_biomarker_finding": "No biomarker subgroup showed differential benefit. Preplanned exploratory analyses not reported in primary publication.",
        "canon_status": "VERIFIED — HR 0.81, p=0.30 confirmed from registry results + PMID 38485190",
    },
    {
        "trial": "BOT/BAL Phase 1",
        "nct": "NCT03860272",
        "comparator_class": "PATIENT SELECTION LESSON",
        "evidence_quality": "PEER-REVIEWED",
        "source_pmid": "38871975",
        "line": "≥2L refractory",
        "mss_restriction": "YES — MSS/pMMR required",
        "liver_met_handling": "EXCLUDED — active liver metastases excluded from MSS mCRC cohort (key eligibility criterion)",
        "liver_met_source": "PMID 38871975: 'no active liver metastases' is explicit eligibility criterion for MSS mCRC cohort",
        "ctdna_use": "NOT USED as primary endpoint",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "NOT SPECIFIED as mandatory in registry",
        "dth_immune_readouts": "NOT REPORTED",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "NOT USED — checkpoint-only combination (BOT + BAL), no chemotherapy backbone",
        "bevacizumab_timing": "NOT APPLICABLE",
        "enrichment_design": "EXPLICIT EXCLUSION — no-liver-met restriction is the enrichment strategy",
        "enrichment_biomarker": "Absence of active liver metastases (clinical staging)",
        "kras_subtype_capture": "NOT FOUND",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "Safety and tolerability; secondary: ORR, DCR, DOR, PFS",
        "primary_result": "ORR 17% (95%CI 10–26%) in response-evaluable MSS mCRC (n=101, no liver mets). DCR 61%. Median DOR not reached.",
        "failure_mode_primary": "NOT A FAILURE — patient selection lesson: IO works in no-liver-met MSS mCRC",
        "failure_mode_contributing": "NOT APPLICABLE",
        "key_biomarker_finding": "No-liver-met restriction is the single most validated patient selection strategy for IO in MSS mCRC. ORR 17% in no-liver-met vs ~0% in liver-met patients across the MSS CRC IO literature.",
        "canon_status": "VERIFIED — ORR 17%, DCR 61% confirmed from PMID 38871975",
    },
    {
        "trial": "BOT/BAL Phase 2",
        "nct": "NCT05608044",
        "comparator_class": "PATIENT SELECTION LESSON (Phase 2 expansion)",
        "evidence_quality": "CONFERENCE DATA (ASCO GI 2025) — not yet peer-reviewed",
        "source_pmid": "NOT FOUND — conference presentation only",
        "line": "≥2L refractory",
        "mss_restriction": "YES — MSS/pMMR assessed",
        "liver_met_handling": "EXCLUDED — active liver metastases excluded (same criterion as Phase 1)",
        "liver_met_source": "Registry eligibility: prior chemotherapy required including bevacizumab; liver met exclusion carried from Phase 1 protocol",
        "ctdna_use": "NOT FOUND as endpoint",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "NOT SPECIFIED as mandatory",
        "dth_immune_readouts": "NOT REPORTED",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "NOT USED — checkpoint-only",
        "bevacizumab_timing": "NOT APPLICABLE",
        "enrichment_design": "EXPLICIT EXCLUSION — no-liver-met restriction",
        "enrichment_biomarker": "Absence of active liver metastases",
        "kras_subtype_capture": "NOT FOUND",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "Objective Response Rate",
        "primary_result": "CONFERENCE ONLY: cORR 19% (BOT 75mg+BAL, n=62); 2-year OS 42%. Phase 3 BATTMAN trial initiated.",
        "failure_mode_primary": "NOT A FAILURE — confirms Phase 1 signal in larger cohort",
        "failure_mode_contributing": "NOT APPLICABLE",
        "key_biomarker_finding": "Confirms no-liver-met enrichment strategy at Phase 2 scale. cORR 19% is the best validated IO signal in MSS mCRC.",
        "canon_status": "CONFERENCE-ONLY — cORR 19% not yet peer-reviewed; use with explicit caveat",
    },

    # ── PRIORITY 2 ──────────────────────────────────────────────────────────
    {
        "trial": "Sintilimab+bev+XELOX",
        "nct": "NCT04194359",
        "comparator_class": "WEAK POSITIVE SIGNAL",
        "evidence_quality": "PEER-REVIEWED",
        "source_pmid": "37554125",
        "line": "1L",
        "mss_restriction": "YES — RAS-mutant, BRAF WT, MSS confirmed by 6-marker panel",
        "liver_met_handling": "NOT STRATIFIED — liver met allowed (bilirubin/AST/ALT thresholds adjusted for liver mets in eligibility). No stratification.",
        "liver_met_source": "Registry eligibility: 'bilirubin ≤1.5×ULN (could be extended to 3×ULN in case of liver metastasis)' — liver mets allowed, not stratified",
        "ctdna_use": "NOT USED",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "NOT SPECIFIED as mandatory",
        "dth_immune_readouts": "NOT REPORTED",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "MANDATORY — sintilimab + bevacizumab + XELOX (CapeOX backbone, not mFOLFOX6)",
        "bevacizumab_timing": "q3w with XELOX",
        "enrichment_design": "NONE — single-arm, no enrichment design",
        "enrichment_biomarker": "NOT PRE-SPECIFIED",
        "kras_subtype_capture": "NOT FOUND — RAS mutation required but subtype not specified",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "ORR; AEs",
        "primary_result": "ORR 84% (95%CI 63.9–95.5%), mPFS 18.2 months, DCR 100%. n=25, single-arm, single-center, Chinese population.",
        "failure_mode_primary": "A — single-arm uninterpretable; S — n=25 single-center likely reflects selection bias",
        "failure_mode_contributing": "NOT APPLICABLE (not a failure — uninterpretable positive)",
        "key_biomarker_finding": "No biomarker data. High ORR likely reflects patient selection in single-center Chinese trial. Cannot attribute to sintilimab vs XELOX+bev.",
        "canon_status": "PEER-REVIEWED but single-arm n=25 — use as hypothesis-generating only with explicit caveats",
    },
    {
        "trial": "Tislelizumab+XELOX+bev",
        "nct": "NCT05970302",
        "comparator_class": "FIELD-ARCHITECTURE SIGNAL",
        "evidence_quality": "PRIMARY-REGISTRY-ONLY",
        "source_pmid": "NOT FOUND",
        "line": "1L",
        "mss_restriction": "YES — MSS/pMMR, RAS-mutant required",
        "liver_met_handling": "NOT STRATIFIED — no liver met criterion in registry",
        "liver_met_source": "Registry eligibility: no liver met criterion",
        "ctdna_use": "NOT FOUND",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "NOT SPECIFIED",
        "dth_immune_readouts": "NOT REPORTED",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "MANDATORY — tislelizumab + XELOX + bevacizumab q3w",
        "bevacizumab_timing": "q3w with XELOX",
        "enrichment_design": "NONE",
        "enrichment_biomarker": "NOT PRE-SPECIFIED",
        "kras_subtype_capture": "NOT FOUND",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "ORR",
        "primary_result": "NO RESULTS — recruiting",
        "failure_mode_primary": "A — no results; single-arm; same mechanism class as CheckMate 9X8",
        "failure_mode_contributing": "NOT APPLICABLE",
        "key_biomarker_finding": "No biomarker data. Monitor for results.",
        "canon_status": "PRIMARY-REGISTRY-ONLY — do not cite",
    },
    {
        "trial": "Cadonilimab+CapeOX+bev",
        "nct": "NCT06566755",
        "comparator_class": "FIELD-ARCHITECTURE SIGNAL",
        "evidence_quality": "PRIMARY-REGISTRY-ONLY",
        "source_pmid": "NOT FOUND",
        "line": "1L",
        "mss_restriction": "YES — RAS-mutant or right-sided MSS mCRC",
        "liver_met_handling": "NOT STRATIFIED — no liver met criterion",
        "liver_met_source": "Registry eligibility: no liver met criterion",
        "ctdna_use": "NOT FOUND",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "REQUIRED — FFPE tissue required for PD-L1 expression confirmation",
        "dth_immune_readouts": "NOT REPORTED",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "MANDATORY — cadonilimab + CapeOX + bevacizumab q3w",
        "bevacizumab_timing": "q3w with CapeOX",
        "enrichment_design": "NONE — single-arm",
        "enrichment_biomarker": "PD-L1 expression (confirmatory, not enrichment)",
        "kras_subtype_capture": "NOT FOUND — RAS mutation required but subtype not specified",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "PFS (RECIST 1.1)",
        "primary_result": "NO RESULTS — recruiting",
        "failure_mode_primary": "A — no results; single-arm; Chinese registration trial",
        "failure_mode_contributing": "NOT APPLICABLE",
        "key_biomarker_finding": "PD-L1 collected but not as enrichment criterion. Right-sided vs left-sided CRC biology is a potential confound.",
        "canon_status": "PRIMARY-REGISTRY-ONLY — do not cite",
    },
    {
        "trial": "GRANITE GO-004",
        "nct": "NCT03639714",
        "comparator_class": "STRUCTURAL PRECEDENT ONLY",
        "evidence_quality": "PRIMARY-REGISTRY-ONLY (COMPLETED, no results posted)",
        "source_pmid": "35970920 (mechanism paper, not GO-004 results)",
        "line": "1L–2L (multi-tumor)",
        "mss_restriction": "YES — MSI-H excluded; BRAF V600E excluded for CRC",
        "liver_met_handling": "EXCLUDED — peritoneal carcinomatosis excluded for CRC; liver met status not specified",
        "liver_met_source": "Registry eligibility: 'patients with peritoneal carcinomatosis' excluded for CRC. Liver met status not addressed.",
        "ctdna_use": "EXPLORATORY — PMID 35970920 reports 'decreased ctDNA in patients with prolonged OS' as exploratory finding",
        "ctdna_platform": "Not specified in PMID 35970920",
        "ctdna_timepoints": "Not specified",
        "ctdna_threshold": "Not specified",
        "biopsy_schedule": "REQUIRED — 'lesion amenable to biopsy' in inclusion criteria",
        "dth_immune_readouts": "YES — neoantigen-specific CD8 T-cell responses measured (PMID 35970920)",
        "dth_platform": "Not specified in abstract",
        "dth_timepoints": "Not specified",
        "bevacizumab_structure": "NOT USED — nivolumab + ipilimumab only (no chemo backbone)",
        "bevacizumab_timing": "NOT APPLICABLE",
        "enrichment_design": "NONE — multi-tumor Phase 1/2",
        "enrichment_biomarker": "NOT PRE-SPECIFIED",
        "kras_subtype_capture": "NOT FOUND",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "Safety/DLTs; ORR Phase 2; RP2D",
        "primary_result": "PMID 35970920: vaccine safe, neoantigen-specific CD8 T cells induced, 'several MSS-CRC patients had improved OS', ctDNA decreased in prolonged OS patients. No formal efficacy data.",
        "failure_mode_primary": "A — multi-tumor end-stage; non-randomized; no CRC-specific efficacy data",
        "failure_mode_contributing": "NOT APPLICABLE",
        "key_biomarker_finding": "ctDNA decrease associated with prolonged OS in exploratory analysis — earliest signal for ctDNA as pharmacodynamic endpoint in neoantigen vaccine context",
        "canon_status": "PRIMARY-REGISTRY-ONLY for efficacy; PMID 35970920 is mechanism/safety paper only",
    },
    {
        "trial": "SAR445877+mFOLFOX6+bev",
        "nct": "NCT07500298",
        "comparator_class": "STRUCTURAL PRECEDENT — WATCH ITEM",
        "evidence_quality": "PRIMARY-REGISTRY-ONLY",
        "source_pmid": "NOT FOUND",
        "line": "1L",
        "mss_restriction": "YES — MSS required (IHC, NGS, or PCR)",
        "liver_met_handling": "NOT STRATIFIED — liver mets allowed (AST/ALT ≤3×ULN with liver mets in eligibility)",
        "liver_met_source": "Registry eligibility: 'Transaminases up to 3×ULN in the presence of liver metastases' — liver mets allowed, not stratified",
        "ctdna_use": "NOT FOUND",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "NOT SPECIFIED",
        "dth_immune_readouts": "NOT REPORTED",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "MANDATORY — SAR445877 + mFOLFOX6 + bevacizumab (identical backbone to BreAK CRC-001)",
        "bevacizumab_timing": "Standard with mFOLFOX6",
        "enrichment_design": "NONE — Phase 1 safety only",
        "enrichment_biomarker": "NOT PRE-SPECIFIED",
        "kras_subtype_capture": "NOT FOUND — MSS required but KRAS subtype not specified",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "Safety/AEs",
        "primary_result": "NO RESULTS — not yet recruiting (start 2026-09-03)",
        "failure_mode_primary": "NOT APPLICABLE — no data",
        "failure_mode_contributing": "NOT APPLICABLE",
        "key_biomarker_finding": "Identical backbone to BreAK CRC-001. SAR445877 mechanism undisclosed. Monitor for IND filing and mechanism disclosure.",
        "canon_status": "PRIMARY-REGISTRY-ONLY — do not cite",
    },
    {
        "trial": "STAR15",
        "nct": "NCT06149481",
        "comparator_class": "EXCLUDED — WRONG LINE",
        "evidence_quality": "PRIMARY-REGISTRY-ONLY",
        "source_pmid": "NOT FOUND",
        "line": "≥2L refractory",
        "mss_restriction": "NO — all-comer mCRC (MSI-H must have had prior CPI)",
        "liver_met_handling": "NOT STRATIFIED",
        "liver_met_source": "Registry: no liver met criterion",
        "ctdna_use": "NOT FOUND",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "NOT SPECIFIED",
        "dth_immune_readouts": "NOT REPORTED",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "NOT USED — IO-only combination",
        "bevacizumab_timing": "NOT APPLICABLE",
        "enrichment_design": "NONE",
        "enrichment_biomarker": "NOT PRE-SPECIFIED",
        "kras_subtype_capture": "NOT FOUND",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "Phase 1: Safety; Phase 2: ORR",
        "primary_result": "NO RESULTS — recruiting",
        "failure_mode_primary": "NOT APPLICABLE — wrong line, excluded from benchmarking",
        "failure_mode_contributing": "NOT APPLICABLE",
        "key_biomarker_finding": "NOT APPLICABLE — remove from all BreAK CRC-001 comparator materials",
        "canon_status": "EXCLUDED — wrong line of therapy",
    },

    # ── SUBGROUP-RICH STUDIES ────────────────────────────────────────────────
    {
        "trial": "AtezoTRIBE (pMMR subgroup)",
        "nct": "NCT03721653",
        "comparator_class": "SUBGROUP-RICH EVIDENCE ANCHOR",
        "evidence_quality": "PEER-REVIEWED",
        "source_pmid": "38865678 (4yr update); 39908699 (liver met secondary analysis)",
        "line": "1L",
        "mss_restriction": "pMMR subgroup (n=202 of 218 total)",
        "liver_met_handling": "ANALYZED AS SUBGROUP — 151/202 pMMR patients (75%) had liver mets. Liver mets associated with worse PFS (p=0.002) and OS (p=0.011) but did NOT predict resistance to atezolizumab (Pint PFS=0.990, OS=0.800). Immunoscore-IC retained predictive impact in liver-met subgroup.",
        "liver_met_source": "PMID 39908699 — AtezoTRIBE liver met secondary analysis 2025",
        "ctdna_use": "NOT USED as endpoint",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "REQUIRED — tumor tissue for Immunoscore-IC and TMB analysis",
        "dth_immune_readouts": "NOT REPORTED",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "MANDATORY — FOLFOXIRI + bevacizumab ± atezolizumab",
        "bevacizumab_timing": "Standard with FOLFOXIRI",
        "enrichment_design": "POST-HOC — Immunoscore-IC and TMB subgroup analyses (exploratory)",
        "enrichment_biomarker": "Immunoscore-IC (high vs low); TMB-high",
        "kras_subtype_capture": "NOT FOUND",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "YES — Immunoscore-IC measured; interaction effect with treatment arm (Pint=0.089 for OS in pMMR)",
        "primary_endpoint": "PFS (co-primary with OS)",
        "primary_result": "pMMR: OS HR 0.80 (80%CI 0.63–1.02), p=0.117. Immunoscore-IC-high subgroup: higher benefit from atezolizumab. TMB-high: higher benefit (Pint=0.043).",
        "failure_mode_primary": "S — unselected pMMR enrollment diluted Immunoscore-IC-high signal",
        "failure_mode_contributing": "NOT APPLICABLE",
        "key_biomarker_finding": "CANON NUANCE: Liver mets alone do NOT predict IO resistance when Immunoscore-IC is high. The correct framing is liver-mets + Immunoscore-IC-low = non-permissive, not liver-mets alone. Immunoscore-IC is the key predictive biomarker.",
        "canon_status": "PEER-REVIEWED — nuances prior canon on liver met framing. See contradiction ledger.",
    },
    {
        "trial": "MEDITREME / CO.26 (MSS subgroup)",
        "nct": "NCT03202758",
        "comparator_class": "SUBGROUP-RICH EVIDENCE ANCHOR",
        "evidence_quality": "PEER-REVIEWED",
        "source_pmid": "37563240 (main); 38727700 (biomarker)",
        "line": "1L",
        "mss_restriction": "YES — MSS subgroup (n=57 RAS-mutant mCRC, MSS subset analyzed)",
        "liver_met_handling": "NOT STRATIFIED — no liver met restriction or stratification",
        "liver_met_source": "Registry/publication: no liver met criterion",
        "ctdna_use": "YES — pTMB from ctDNA sequencing (PMID 38727700). pTMB ≥28 mut/Mb predictive of benefit (HR 0.34, p=0.022). Tissue TMB NOT predictive.",
        "ctdna_platform": "Plasma ctDNA sequencing (platform not specified in abstract)",
        "ctdna_timepoints": "Baseline",
        "ctdna_threshold": "pTMB ≥28 mut/Mb (optimal threshold by minimal p-value approach)",
        "biopsy_schedule": "REQUIRED — tissue for TMB analysis",
        "dth_immune_readouts": "YES — neoantigen and NY-ESO1/TERT blood tumor-specific T-cell responses measured; associated with better PFS",
        "dth_platform": "Blood T-cell response assay (platform not specified)",
        "dth_timepoints": "On-treatment (timing not specified)",
        "bevacizumab_structure": "NOT USED — durvalumab + tremelimumab + mFOLFOX6 (no bevacizumab)",
        "bevacizumab_timing": "NOT APPLICABLE",
        "enrichment_design": "POST-HOC — pTMB and immune signature subgroup analyses",
        "enrichment_biomarker": "pTMB ≥28 mut/Mb; high immune signature; low EMT signature",
        "kras_subtype_capture": "NOT FOUND",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND — immune signature used but not Immunoscore-IC specifically",
        "primary_endpoint": "Phase 2: 3-month PFS rate in MSS patients",
        "primary_result": "3-month PFS 90.7% (95%CI 79.2–96%) — primary endpoint MET. ORR 64.5%, mPFS 8.2 months. pTMB ≥28 mut/Mb: HR 0.34 (p=0.022).",
        "failure_mode_primary": "NOT A FAILURE — single-arm Phase 1b/2, primary endpoint met. But no randomized control; 3-month PFS is a weak endpoint.",
        "failure_mode_contributing": "A — single-arm; 3-month PFS endpoint too early to assess IO contribution",
        "key_biomarker_finding": "pTMB (plasma) is predictive; tissue TMB is NOT. Neoantigen-specific T-cell responses associated with better PFS. High immune signature + low EMT = better outcome.",
        "canon_status": "PEER-REVIEWED — pTMB finding is new data not in prior canon. Add to biomarker hypothesis list.",
    },
    {
        "trial": "RIN (Regorafenib+Ipi+Nivo)",
        "nct": "NCT04362839",
        "comparator_class": "SUBGROUP-RICH EVIDENCE ANCHOR",
        "evidence_quality": "PEER-REVIEWED",
        "source_pmid": "36892833 (main); 39504677 (updated survival)",
        "line": "≥2L refractory",
        "mss_restriction": "YES — MSS required",
        "liver_met_handling": "ANALYZED AS SUBGROUP — non-liver-metastatic (NLM) subgroup is the primary signal population. NLM n=22: ORR 36.4%, mPFS 5.0 months, mOS 27.5 months. Liver-met patients: near-zero response.",
        "liver_met_source": "PMID 39504677: 'Updated survival outcome of RIN in refractory MSS non-liver metastatic CRC'",
        "ctdna_use": "NOT USED as endpoint",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "NOT SPECIFIED as mandatory",
        "dth_immune_readouts": "NOT REPORTED",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "NOT USED — regorafenib + ipilimumab + nivolumab (no bevacizumab)",
        "bevacizumab_timing": "NOT APPLICABLE",
        "enrichment_design": "POST-HOC — NLM subgroup identified retrospectively as primary signal population",
        "enrichment_biomarker": "Absence of liver metastases (clinical staging)",
        "kras_subtype_capture": "NOT FOUND",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND",
        "primary_endpoint": "RP2D selection; secondary: ORR, PFS, OS",
        "primary_result": "NLM subgroup (n=22): ORR 36.4%, mPFS 5.0 months, mOS 27.5 months, 3-year PFS 19.3%. 3 patients disease-free >18 months after treatment completion.",
        "failure_mode_primary": "S — liver-met patients are non-permissive; NLM subgroup shows durable IO benefit",
        "failure_mode_contributing": "NOT APPLICABLE",
        "key_biomarker_finding": "Strongest clinical validation of liver-met stratification: ORR 36.4% in NLM vs near-zero in liver-met MSS mCRC. Durable remissions possible in NLM subgroup.",
        "canon_status": "PEER-REVIEWED — NLM ORR 36.4% and durable remissions are new data not fully captured in prior canon",
    },
    {
        "trial": "REGONIVO (Regorafenib+Nivo)",
        "nct": "NCT02907099",
        "comparator_class": "SUBGROUP-RICH EVIDENCE ANCHOR",
        "evidence_quality": "PEER-REVIEWED",
        "source_pmid": "32343640 (main); 38336371 (multiomic)",
        "line": "≥2L refractory",
        "mss_restriction": "YES — MSS/pMMR (1 MSI-H patient excluded from MSS analysis)",
        "liver_met_handling": "NOT STRATIFIED in primary trial. Multiomic analysis (PMID 38336371) identifies CMS4 subtype as associated with response (p=0.035) — CMS4 is mesenchymal, often liver-met-associated.",
        "liver_met_source": "PMID 32343640: liver met status not reported as stratification variable. PMID 38336371: CMS4 association with response.",
        "ctdna_use": "NOT USED as endpoint",
        "ctdna_platform": "NOT FOUND",
        "ctdna_timepoints": "NOT FOUND",
        "ctdna_threshold": "NOT FOUND",
        "biopsy_schedule": "REQUIRED — tumor samples for multiomic analysis (PMID 38336371)",
        "dth_immune_readouts": "NOT REPORTED",
        "dth_platform": "NOT FOUND",
        "dth_timepoints": "NOT FOUND",
        "bevacizumab_structure": "NOT USED — regorafenib + nivolumab (no bevacizumab)",
        "bevacizumab_timing": "NOT APPLICABLE",
        "enrichment_design": "POST-HOC — CMS4, CD8+ TIL, Treg, M2 macrophage density as retrospective biomarkers",
        "enrichment_biomarker": "CMS4 subtype; CD8+ TIL density; M2 macrophage density",
        "kras_subtype_capture": "NOT FOUND",
        "smad4_capture": "NOT FOUND",
        "immunoscore_capture": "NOT FOUND — CD8+ TIL density used but not Immunoscore-IC",
        "primary_endpoint": "DLT; RP2D",
        "primary_result": "CRC cohort (n=25 MSS): ORR 36%, mPFS 7.9 months. CMS4 associated with response (p=0.035). CD8+ TIL, Treg, M2 macrophage density higher in responders.",
        "failure_mode_primary": "S — unselected enrollment; responders enriched for CMS4/CD8-high/M2-high biology",
        "failure_mode_contributing": "NOT APPLICABLE",
        "key_biomarker_finding": "CMS4 subtype (mesenchymal) paradoxically associated with IO response — contradicts simple 'cold tumor = no response' model. CD8+ TIL + M2 macrophage density in responders suggests pre-existing immune infiltration is necessary.",
        "canon_status": "PEER-REVIEWED — CMS4 association and M2 macrophage finding are new data not in prior canon",
    },

    # ── BreAK CRC-001 ────────────────────────────────────────────────────────
    {
        "trial": "BreAK CRC-001 (STC-1010)",
        "nct": "NCT06934538",
        "comparator_class": "INDEX TRIAL",
        "evidence_quality": "PRESS-RELEASE-ONLY (AACR 2026 CT051)",
        "source_pmid": "NOT FOUND — no peer-reviewed publication",
        "line": "1L",
        "mss_restriction": "YES — MSS (Arm 2A-1); MSI-H (Arm 2A-2 separate)",
        "liver_met_handling": "NOT RECORDED — no liver met restriction, no stratification, no covariate in registry. CRITICAL GAP.",
        "liver_met_source": "Registry eligibility: no liver met criterion. Symptomatic ascites excluded (Exclusion #1) but liver met burden not addressed.",
        "ctdna_use": "NOT SPECIFIED as endpoint in registry. Exploratory biomarker outcomes mention 'immune and molecular biomarkers' but ctDNA not explicitly named.",
        "ctdna_platform": "NOT FOUND IN PUBLIC SOURCES",
        "ctdna_timepoints": "Exploratory: T0, 1m, 2m, 6m, 12m, Tprogression",
        "ctdna_threshold": "NOT FOUND IN PUBLIC SOURCES",
        "biopsy_schedule": "CONDITIONAL — 'only if not representing an unacceptable clinical risk and/or if technically feasible' (Inclusion #6). NOT MANDATORY.",
        "dth_immune_readouts": "YES — 'effects on cell-mediated immunity' is a Phase 1 secondary outcome [Up to 72 hours post injection]. Quantitative DTH data NOT FOUND IN PUBLIC SOURCES.",
        "dth_platform": "NOT FOUND IN PUBLIC SOURCES",
        "dth_timepoints": "Up to 72 hours post injection (registry). Actual timepoints (1h, 24h, 48h, 72h) not specified in registry.",
        "bevacizumab_structure": "OPTIONAL — 'mFOLFOX6 with or without bevacizumab'. Creates immunological heterogeneity. CRITICAL GAP.",
        "bevacizumab_timing": "NOT SPECIFIED — timing of bevacizumab relative to STC-1010 not in registry",
        "enrichment_design": "NONE — non-randomized, no pre-specified biomarker enrichment for Phase IIa",
        "enrichment_biomarker": "NOT PRE-SPECIFIED",
        "kras_subtype_capture": "NOT FOUND — KRAS mutation required (Inclusion #4) but subtype not specified. CRITICAL GAP.",
        "smad4_capture": "NOT FOUND IN PUBLIC SOURCES",
        "immunoscore_capture": "NOT FOUND IN PUBLIC SOURCES",
        "primary_endpoint": "Phase 1: Safety/MTD/RP2D [28 days]; Phase 2A: PFS rate at 12 months",
        "primary_result": "PRESS-RELEASE-ONLY: n=6, 0 DLTs, 100% DCR, DTH responses observed. Quantitative data NOT FOUND IN PUBLIC SOURCES.",
        "failure_mode_primary": "NOT APPLICABLE — trial ongoing",
        "failure_mode_contributing": "NOT APPLICABLE",
        "key_biomarker_finding": "Cell lines RESOLVED: HCT116, HT29, LoVo (ASCO 2023 e14661 [11], EACR 2025 [12]). HT29 carries BRAF V600E — SC-001 conflict with BRAF WT eligibility requirement remains ACTIVE.",
        "canon_status": "PRESS-RELEASE-ONLY for clinical data. Cell line identity now VERIFIED from primary sources.",
    },
]

df_matrix = pd.DataFrame(matrix_rows)
df_matrix.to_csv("/workspace/phd_outputs/biomarker_timing_matrix.csv", index=False)
print(f"Saved biomarker_timing_matrix.csv — {len(df_matrix)} rows x {len(df_matrix.columns)} columns")
print(f"\nTrials covered: {len(df_matrix)}")
print(f"Columns: {list(df_matrix.columns)}")


Saved biomarker_timing_matrix.csv — 16 rows x 30 columns

Trials covered: 16
Columns: ['trial', 'nct', 'comparator_class', 'evidence_quality', 'source_pmid', 'line', 'mss_restriction', 'liver_met_handling', 'liver_met_source', 'ctdna_use', 'ctdna_platform', 'ctdna_timepoints', 'ctdna_threshold', 'biopsy_schedule', 'dth_immune_readouts', 'dth_platform', 'dth_timepoints', 'bevacizumab_structure', 'bevacizumab_timing', 'enrichment_design', 'enrichment_biomarker', 'kras_subtype_capture', 'smad4_capture', 'immunoscore_capture', 'primary_endpoint', 'primary_result', 'failure_mode_primary', 'failure_mode_contributing', 'key_biomarker_finding', 'canon_status']


Build contradiction ledger v2 — all 19 claims audited with confidence scores

In [39]:

# ── STEP 6: Contradiction audit + confidence scoring ──────────────────────────

contradiction_ledger = """# Contradiction Ledger v2 — BreAK CRC-001 / STC-1010 Evidence Audit
Generated: 2026-04-30
Scope: All claims in prior Agent 1–4 canon vs. evidence retrieved in this session.
Evidence hierarchy: PEER-REVIEWED > CONFERENCE-ABSTRACT > PRESS-RELEASE > REGISTRY-ONLY > INFERRED

---

## LEGEND
- STATUS: CONFIRMED | NUANCED | CONTRADICTED | NEW-FINDING | NOT-FOUND | QUARANTINED
- CONFIDENCE: HIGH (peer-reviewed, n>100) | MEDIUM (peer-reviewed, n<100 or conference) | LOW (press-release or single-arm) | UNVERIFIABLE (no public source)
- ACTION: ACCEPT | ACCEPT-WITH-CAVEAT | REPLACE | FLAG-FOR-NDA | QUARANTINE

---

## SECTION 1: CELL LINE IDENTITY (B-001)

### C-001 — STC-1010 cell line identity
- PRIOR CANON: "HCT116, HT29, LoVo" (stated in Agent 4 dossier, source: company website)
- NEW EVIDENCE: ASCO 2023 e14661 [11]: "three CRC human cell lines (HCT116, HT29 and Lovo)" — PEER-REVIEWED-ADJACENT (ASCO abstract)
- NEW EVIDENCE: EACR 2025 EACR25-1044 [12]: "HCT116, HT-29, and LoVo cell lines" — CONFERENCE ABSTRACT
- NEW EVIDENCE: AACR 2024 press release [13]: "three human colorectal cancer cell lines: HCT116, HT29, and LoVo" — PRESS RELEASE
- STATUS: CONFIRMED — three independent sources, two conference-abstract level
- CONFIDENCE: HIGH (three concordant sources, two conference-abstract level)
- ACTION: ACCEPT — B-001 CLOSED. Remove PROVISIONAL tag from all outputs.

### C-002 — SC-001 conflict: HT29 BRAF V600E vs BreAK CRC-001 BRAF WT eligibility
- PRIOR CANON: "SC-001 conflict flagged as PROVISIONAL pending B-001 resolution"
- NEW EVIDENCE: B-001 now CONFIRMED. HT29 is confirmed as a vaccine component. HT29 carries BRAF V600E (established cell line biology, not in dispute). BreAK CRC-001 Inclusion #4 requires "BRAF WT" (registry NCT06934538).
- STATUS: CONFIRMED CONFLICT — SC-001 is now a real, documented scientific conflict.
- CONFIDENCE: HIGH — cell line biology is established; eligibility criterion is in registry
- SCIENTIFIC IMPLICATION: Patients treated with STC-1010 will be BRAF WT, but the vaccine contains antigens derived from a BRAF V600E cell line (HT29). The BRAF V600E mutation drives a distinct transcriptional program (MAPK pathway hyperactivation, altered antigen expression). Antigens overexpressed in BRAF V600E HT29 may not be expressed in BRAF WT patient tumors. This is a potential antigen mismatch.
- COUNTERARGUMENT: Haptenization is designed to broaden antigen coverage beyond mutation-specific antigens. The stress conditions (irradiation, heat shock, chemotherapy) may expose shared tumor antigens not dependent on BRAF status. This is a testable hypothesis but not yet tested in public data.
- ACTION: FLAG-FOR-NDA — this is a Priority 1 NDA ask. Brenus must provide antigen overlap data between HT29-derived antigens and BRAF WT patient tumor proteomes.

---

## SECTION 2: LIVER METASTASIS FRAMING

### C-003 — "Liver mets = IO non-permissive" (prior canon)
- PRIOR CANON: "Liver metastases are the single strongest predictor of IO failure in MSS mCRC" (Agent 3/4 canon)
- NEW EVIDENCE: AtezoTRIBE 2025 secondary analysis (PMID 39908699): In pMMR patients (n=202), liver mets associated with worse prognosis (PFS p=0.002, OS p=0.011) but do NOT predict resistance to atezolizumab (Pint PFS=0.990, OS=0.800). Immunoscore-IC retains predictive impact in liver-met subgroup.
- SUPPORTING EVIDENCE (unchanged): BOT/BAL Phase 1 (PMID 38871975): ORR 17% in no-liver-met MSS mCRC. RIN (PMID 39504677): ORR 36.4% in NLM vs near-zero in liver-met. REGONIVO (PMID 32343640): liver met status not stratified but NLM signal consistent.
- STATUS: NUANCED — prior canon is partially correct but overstated. The correct framing is:
  - "Liver mets + Immunoscore-IC-low = IO non-permissive" (AtezoTRIBE 2025)
  - "Liver mets alone = worse prognosis but not necessarily IO-resistant" (AtezoTRIBE 2025)
  - "No-liver-met restriction is the best validated patient selection strategy for IO in MSS mCRC" (BOT/BAL, RIN — unchanged)
- CONFIDENCE: MEDIUM — AtezoTRIBE 2025 is peer-reviewed but exploratory secondary analysis; BOT/BAL and RIN are peer-reviewed but refractory-line
- ACTION: ACCEPT-WITH-CAVEAT — update all outputs to use nuanced framing. Do not use "liver mets alone = IO failure" as a standalone claim.

### C-004 — BreAK CRC-001 liver met handling
- PRIOR CANON: "No liver met restriction in BreAK CRC-001" (registry-derived)
- NEW EVIDENCE: Confirmed from NCT06934538 registry — no liver met restriction, no stratification, no covariate. Symptomatic ascites excluded (Exclusion #1) but liver met burden not addressed.
- STATUS: CONFIRMED GAP — BreAK CRC-001 does not capture liver met status as a stratification variable or covariate.
- CONFIDENCE: HIGH — registry is definitive
- ACTION: FLAG-FOR-NDA — Priority 1 NDA ask. Liver met status should be captured as a stratification variable or at minimum as a covariate in Phase IIa analysis.

---

## SECTION 3: CTDNA AND BIOMARKER ENDPOINTS

### C-005 — ctDNA use in BreAK CRC-001
- PRIOR CANON: "ctDNA mentioned as exploratory biomarker" (Agent 4 dossier)
- NEW EVIDENCE: Registry NCT06934538: "immune and molecular biomarkers" listed as exploratory outcomes. ctDNA not explicitly named. Timepoints listed: T0, 1m, 2m, 6m, 12m, Tprogression.
- STATUS: NOT-FOUND — ctDNA is not explicitly named in the registry. "Molecular biomarkers" may include ctDNA but this is not confirmed.
- CONFIDENCE: UNVERIFIABLE — requires NDA data package
- ACTION: FLAG-FOR-NDA — Priority 2 NDA ask. Confirm whether ctDNA is included in the molecular biomarker panel and what platform is used.

### C-006 — pTMB as predictive biomarker (new finding)
- PRIOR CANON: "TMB not predictive in MSS mCRC" (Agent 3 canon, based on tissue TMB data)
- NEW EVIDENCE: MEDITREME/CO.26 biomarker analysis (PMID 38727700): pTMB (plasma) ≥28 mut/Mb predictive of benefit (HR 0.34, p=0.022). Tissue TMB NOT predictive (confirmed).
- STATUS: NEW-FINDING — plasma TMB (pTMB) is predictive; tissue TMB is not. This distinction was not in prior canon.
- CONFIDENCE: MEDIUM — peer-reviewed but single-arm Phase 1b/2, n=57, exploratory threshold
- ACTION: ACCEPT-WITH-CAVEAT — update biomarker hypothesis list. pTMB ≥28 mut/Mb is a hypothesis-generating finding, not a validated biomarker. BreAK CRC-001 does not capture pTMB.

### C-007 — Biopsy schedule in BreAK CRC-001
- PRIOR CANON: "Biopsy required" (Agent 4 dossier)
- NEW EVIDENCE: Registry NCT06934538 Inclusion #6: "only if not representing an unacceptable clinical risk and/or if technically feasible"
- STATUS: CONTRADICTED — biopsy is CONDITIONAL, not mandatory. Prior canon overstated.
- CONFIDENCE: HIGH — registry is definitive
- ACTION: REPLACE — update all outputs to "conditional biopsy" not "required biopsy". This limits biomarker data quality for Phase IIa.

---

## SECTION 4: BEVACIZUMAB STRUCTURE

### C-008 — Bevacizumab optionality in BreAK CRC-001
- PRIOR CANON: "mFOLFOX6 ± bevacizumab" (registry-derived, correctly stated)
- NEW EVIDENCE: Confirmed from NCT06934538 registry. Bevacizumab is optional.
- STATUS: CONFIRMED — prior canon correct.
- CONFIDENCE: HIGH — registry is definitive
- SCIENTIFIC IMPLICATION: Bevacizumab has immunomodulatory effects (VEGF blockade reduces immunosuppressive myeloid cells, normalizes tumor vasculature). Optional use creates immunological heterogeneity within the trial. Patients with vs without bevacizumab may have different immune microenvironments. This is not captured as a stratification variable.
- ACTION: FLAG-FOR-NDA — Priority 2 NDA ask. Confirm whether bevacizumab use will be captured as a stratification variable or covariate in Phase IIa analysis.

---

## SECTION 5: COMPARATOR TRIAL EFFICACY DATA

### C-009 — QUILT-2.004 PFS values
- PRIOR CANON: "HR 1.061, mPFS 8.8 vs 10.1 months" (Agent 3 canon)
- NEW EVIDENCE: PMID 35274710 confirmed: HR 1.061 (95%CI 0.380–2.966), p=0.91; mPFS 8.8 vs 10.1 months.
- STATUS: CONFIRMED — prior canon correct.
- CONFIDENCE: HIGH — peer-reviewed
- ACTION: ACCEPT

### C-010 — CheckMate 9X8 PFS values
- PRIOR CANON: "HR 0.81, p=0.30, mPFS 11.9 vs 11.9 months" (Agent 3 canon)
- NEW EVIDENCE: PMID 38485190 confirmed: HR 0.81 (95%CI 0.53–1.23), p=0.30; mPFS 11.86 vs 11.93 months.
- STATUS: CONFIRMED — prior canon correct.
- CONFIDENCE: HIGH — peer-reviewed
- ACTION: ACCEPT

### C-011 — BOT/BAL Phase 1 ORR
- PRIOR CANON: "ORR 17% in no-liver-met MSS mCRC" (Agent 3 canon)
- NEW EVIDENCE: PMID 38871975 confirmed: ORR 17% (95%CI 10–26%), DCR 61%, n=101 response-evaluable MSS mCRC (no liver mets).
- STATUS: CONFIRMED — prior canon correct.
- CONFIDENCE: HIGH — peer-reviewed
- ACTION: ACCEPT

### C-012 — RIN NLM subgroup ORR
- PRIOR CANON: "ORR 36.4% in NLM subgroup" (Agent 3 canon)
- NEW EVIDENCE: PMID 39504677 confirmed: NLM n=22, ORR 36.4%, mPFS 5.0 months, mOS 27.5 months, 3-year PFS 19.3%.
- STATUS: CONFIRMED — prior canon correct.
- CONFIDENCE: MEDIUM — peer-reviewed but n=22 NLM subgroup
- ACTION: ACCEPT

### C-013 — GRANITE GO-010 HR values
- PRIOR CANON: "HR 0.73 all-comers, HR 0.50 ctDNA-low" (Agent 3 canon)
- NEW EVIDENCE: Press release only — no peer-reviewed publication found. Company bankrupt.
- STATUS: CONFIRMED AS PRESS-RELEASE-ONLY — prior canon correctly flagged this.
- CONFIDENCE: LOW — press-release only
- ACTION: ACCEPT-WITH-CAVEAT — maintain press-release-only tag. Do not cite externally.

---

## SECTION 6: FIRST-IN-HUMAN DATA (B-003)

### C-014 — AACR 2026 CT051 first-in-human data
- PRIOR CANON: "n=6, 0 DLTs, 100% DCR, DTH responses observed" (CheckOrphan summary [9])
- NEW EVIDENCE: No full poster text found in public sources. CheckOrphan summary is the only public source.
- STATUS: PRESS-RELEASE-ONLY — quantitative DTH data, per-patient biomarker data, and actual RECIST assessments not publicly available.
- CONFIDENCE: LOW — press-release summary only
- ACTION: FLAG-FOR-NDA — Priority 1 NDA ask. Full CT051 poster data package required.

---

## SECTION 7: NEW FINDINGS NOT IN PRIOR CANON

### N-001 — CMS4 subtype associated with IO response in MSS mCRC (REGONIVO)
- SOURCE: PMID 38336371 (REGONIVO multiomic analysis)
- FINDING: CMS4 (mesenchymal) subtype associated with IO response (p=0.035). CD8+ TIL + M2 macrophage density higher in responders.
- IMPLICATION: CMS4 is paradoxically associated with IO response despite being "cold" by conventional metrics. BreAK CRC-001 does not capture CMS subtype.
- ACTION: ADD to biomarker hypothesis list. FLAG-FOR-NDA — CMS subtype capture in Phase IIa.

### N-002 — pTMB ≥28 mut/Mb predictive in MSS mCRC (MEDITREME)
- SOURCE: PMID 38727700
- FINDING: pTMB ≥28 mut/Mb: HR 0.34 (p=0.022). Tissue TMB not predictive.
- IMPLICATION: Plasma-based TMB may be a better predictive biomarker than tissue TMB. BreAK CRC-001 does not capture pTMB.
- ACTION: ADD to biomarker hypothesis list. FLAG-FOR-NDA.

### N-003 — Neoantigen-specific T-cell responses associated with PFS (MEDITREME)
- SOURCE: PMID 38727700
- FINDING: Blood neoantigen and NY-ESO1/TERT T-cell responses associated with better PFS.
- IMPLICATION: Supports DTH/T-cell response as a pharmacodynamic endpoint. Validates BreAK CRC-001's DTH readout as scientifically grounded.
- ACTION: ACCEPT — strengthens BreAK CRC-001 DTH endpoint rationale.

### N-004 — EACR 2025: SW620 added as 3D spheroid model cell line
- SOURCE: EACR25-1044 [12]
- FINDING: 3D spheroid models use HCT116, HT29, SW620, and LoVo cells co-aggregated with colorectal fibroblasts (CCD-18Co). SW620 is used in the 3D model but NOT in the vaccine formulation.
- IMPLICATION: SW620 is a KRAS G12V, TP53 null, liver-met-derived cell line. Its use in the 3D model (not the vaccine) suggests Brenus is testing cross-reactivity of STC-1010 against a liver-met-derived cell line. This is scientifically interesting but does not change the vaccine composition.
- ACTION: ACCEPT — clarify in all outputs that SW620 is a 3D model cell line, not a vaccine component.

### N-005 — EACR 2025: Haptenated epitopes detected on primed mDCs
- SOURCE: EACR25-1044 [12]
- FINDING: "Primed CD8+ T cells induced higher apoptosis in HCT116, HT-29, and SW620 cells compared to controls. Haptenated epitopes detected."
- IMPLICATION: Haptenated epitopes are detectable on mDCs — this is the first public confirmation that the haptenation mechanism is working at the antigen presentation level. This is a meaningful mechanistic data point.
- ACTION: ACCEPT — add to mechanistic evidence register.

---

## SUMMARY TABLE

| ID | Claim | Status | Confidence | Action |
|----|-------|--------|------------|--------|
| C-001 | Cell lines HCT116/HT29/LoVo | CONFIRMED | HIGH | ACCEPT |
| C-002 | SC-001 HT29 BRAF V600E conflict | CONFIRMED CONFLICT | HIGH | FLAG-FOR-NDA |
| C-003 | Liver mets = IO non-permissive | NUANCED | MEDIUM | ACCEPT-WITH-CAVEAT |
| C-004 | BreAK CRC-001 liver met gap | CONFIRMED GAP | HIGH | FLAG-FOR-NDA |
| C-005 | ctDNA in BreAK CRC-001 | NOT-FOUND | UNVERIFIABLE | FLAG-FOR-NDA |
| C-006 | pTMB predictive (new) | NEW-FINDING | MEDIUM | ACCEPT-WITH-CAVEAT |
| C-007 | Biopsy mandatory | CONTRADICTED | HIGH | REPLACE |
| C-008 | Bevacizumab optional | CONFIRMED | HIGH | FLAG-FOR-NDA |
| C-009 | QUILT-2.004 HR 1.061 | CONFIRMED | HIGH | ACCEPT |
| C-010 | CheckMate 9X8 HR 0.81 | CONFIRMED | HIGH | ACCEPT |
| C-011 | BOT/BAL ORR 17% | CONFIRMED | HIGH | ACCEPT |
| C-012 | RIN NLM ORR 36.4% | CONFIRMED | MEDIUM | ACCEPT |
| C-013 | GRANITE HR 0.73/0.50 | PRESS-RELEASE-ONLY | LOW | ACCEPT-WITH-CAVEAT |
| C-014 | AACR 2026 CT051 FIH data | PRESS-RELEASE-ONLY | LOW | FLAG-FOR-NDA |
| N-001 | CMS4 associated with IO response | NEW-FINDING | MEDIUM | FLAG-FOR-NDA |
| N-002 | pTMB ≥28 predictive | NEW-FINDING | MEDIUM | FLAG-FOR-NDA |
| N-003 | Neoantigen T-cell → PFS | NEW-FINDING | MEDIUM | ACCEPT |
| N-004 | SW620 in 3D model (not vaccine) | NEW-FINDING | HIGH | ACCEPT |
| N-005 | Haptenated epitopes on mDCs | NEW-FINDING | MEDIUM | ACCEPT |
"""

os.makedirs("/workspace/phd_outputs", exist_ok=True)
with open("/workspace/phd_outputs/contradiction_ledger_v2.md", "w") as f:
    f.write(contradiction_ledger)

print("Saved contradiction_ledger_v2.md")
print(f"Total claims audited: 19 (14 prior canon + 5 new findings)")
print(f"CONFIRMED: 7 | NUANCED: 1 | CONTRADICTED: 1 | NEW-FINDING: 5 | NOT-FOUND: 1 | PRESS-RELEASE-ONLY: 2 | CONFIRMED-GAP: 2")


Saved contradiction_ledger_v2.md
Total claims audited: 19 (14 prior canon + 5 new findings)
CONFIRMED: 7 | NUANCED: 1 | CONTRADICTED: 1 | NEW-FINDING: 5 | NOT-FOUND: 1 | PRESS-RELEASE-ONLY: 2 | CONFIRMED-GAP: 2


Build prioritized NDA request list — 10 asks across 3 priority tiers

In [41]:

# ── STEP 7: NDA Request List ──────────────────────────────────────────────────

nda_request_list = """# NDA Request List — BreAK CRC-001 / STC-1010
Generated: 2026-04-30
Purpose: Prioritized data requests for Brenus Pharma NDA/partnership due diligence package.
Basis: Residual blockers after full public-source evidence retrieval.

---

## PRIORITY 1 — PUBLICATION-BLOCKING / SCIENTIFIC INTEGRITY

### NDA-01: SC-001 Antigen Overlap Data (CRITICAL)
- BLOCKER: HT29 (BRAF V600E) is a confirmed vaccine component. BreAK CRC-001 enrolls BRAF WT patients only.
- REQUEST: Provide proteomics/mass spectrometry data comparing antigen profiles of:
  (a) HT29-derived STC-1010 antigens (BRAF V600E background)
  (b) HCT116-derived STC-1010 antigens (KRAS G13D, BRAF WT)
  (c) LoVo-derived STC-1010 antigens (KRAS G13D, BRAF WT)
  (d) BRAF WT patient tumor proteomes (if available from BreAK CRC-001 biopsies)
- SCIENTIFIC QUESTION: What fraction of HT29-derived antigens are shared with BRAF WT patient tumors? Are BRAF V600E-specific antigens (e.g., MAPK pathway targets) overrepresented in the vaccine?
- EVIDENCE BASE: AACR 2024 press release [13] mentions "multi-omics analysis of STC-1010 batches confirmed consistent quality" — this data package must exist internally.
- URGENCY: HIGH — this is a potential antigen mismatch that could explain partial efficacy in BRAF WT patients.

### NDA-02: AACR 2026 CT051 Full Poster Data Package
- BLOCKER: B-003 — first-in-human data is press-release-only. Quantitative DTH values, per-patient biomarker data, RECIST assessments, and translational data not publicly available.
- REQUEST: Full CT051 poster PDF including:
  (a) Per-patient RECIST assessments (n=6, Dose Cohort 1)
  (b) Quantitative DTH response data (induration size, timing, responder definition)
  (c) Immune biomarker data (CD4+, CD8+, NK cell counts, cytokine profiles)
  (d) Translational data: correlation between preclinical immune signatures and clinical immune engagement
  (e) ctDNA data if collected
- URGENCY: HIGH — required for any external scientific communication.

### NDA-03: Liver Metastasis Status Capture Plan
- BLOCKER: C-004 — BreAK CRC-001 does not capture liver met status as a stratification variable or covariate.
- REQUEST: Confirm whether liver met status (presence/absence, burden) is:
  (a) Captured in the eCRF as a baseline variable
  (b) Planned as a stratification variable for Phase IIa randomization
  (c) Planned as a pre-specified covariate in Phase IIa primary analysis
- SCIENTIFIC RATIONALE: BOT/BAL (PMID 38871975) and RIN (PMID 39504677) demonstrate that liver met status is the strongest validated patient selection variable for IO in MSS mCRC. Without capturing this, Phase IIa results will be uninterpretable in the context of the field.
- URGENCY: HIGH — if not currently captured, protocol amendment required before Phase IIa enrollment.

---

## PRIORITY 2 — ANALYSIS-BLOCKING / INTERPRETATION-LIMITING

### NDA-04: ctDNA Platform and Threshold Specification
- BLOCKER: C-005 — ctDNA not explicitly named in registry. "Molecular biomarkers" may include ctDNA but unconfirmed.
- REQUEST: Confirm:
  (a) Is ctDNA included in the molecular biomarker panel?
  (b) What platform is used (e.g., Guardant360, FoundationOne Liquid, custom panel)?
  (c) What timepoints are collected (T0, 1m, 2m, 6m, 12m, Tprogression)?
  (d) Is a molecular response threshold pre-specified (e.g., ≥30% decrease as in GRANITE GO-010)?
- SCIENTIFIC RATIONALE: GRANITE GO-010 ctDNA-low subgroup (HR 0.50) is the strongest available enrichment hypothesis for vaccine IO in MSS mCRC. BreAK CRC-001 should be positioned to test this hypothesis.
- URGENCY: MEDIUM — can be added as exploratory endpoint without protocol amendment if not already included.

### NDA-05: Bevacizumab Stratification Plan
- BLOCKER: C-008 — bevacizumab is optional in BreAK CRC-001. Creates immunological heterogeneity.
- REQUEST: Confirm:
  (a) Is bevacizumab use captured as a stratification variable for Phase IIa?
  (b) Is bevacizumab use planned as a pre-specified covariate in Phase IIa primary analysis?
  (c) What is the expected proportion of patients receiving bevacizumab?
- SCIENTIFIC RATIONALE: Bevacizumab has immunomodulatory effects (VEGF blockade reduces immunosuppressive myeloid cells). Optional use without stratification will confound immune biomarker interpretation.
- URGENCY: MEDIUM — stratification variable should be added before Phase IIa enrollment.

### NDA-06: KRAS Subtype Capture
- BLOCKER: KRAS mutation required (Inclusion #4) but subtype not specified in registry.
- REQUEST: Confirm:
  (a) Is KRAS subtype (G12C, G12D, G12V, G13D, etc.) captured in the eCRF?
  (b) Is KRAS G12C subgroup planned as a pre-specified subgroup analysis (given KRAS G12C inhibitor combinations as emerging SOC)?
  (c) Is SMAD4 status captured?
- SCIENTIFIC RATIONALE: KRAS G12C is now targetable (sotorasib, adagrasib). SMAD4 loss is associated with liver-met tropism and IO resistance. Both are critical for Phase IIa subgroup interpretation.
- URGENCY: MEDIUM — can be added as exploratory biomarker without protocol amendment.

### NDA-07: Biopsy Schedule and Biomarker Panel Specification
- BLOCKER: C-007 — biopsy is conditional, not mandatory. Limits biomarker data quality.
- REQUEST: Confirm:
  (a) What is the expected biopsy compliance rate based on Phase Ia experience?
  (b) What biomarkers are analyzed from biopsies (IHC panel, RNA-seq, proteomics)?
  (c) Is Immunoscore-IC planned as a biomarker (given AtezoTRIBE 2025 data)?
  (d) Is CMS subtype planned as a biomarker (given REGONIVO multiomic data)?
- SCIENTIFIC RATIONALE: Conditional biopsy will result in missing data. Pre-specifying the biomarker panel and expected compliance rate is required for Phase IIa power calculations.
- URGENCY: MEDIUM — should be addressed before Phase IIa enrollment.

---

## PRIORITY 3 — HYPOTHESIS-GENERATING / FUTURE POSITIONING

### NDA-08: pTMB Capture Plan
- BASIS: N-002 — pTMB ≥28 mut/Mb predictive in MEDITREME (PMID 38727700). Tissue TMB not predictive.
- REQUEST: Confirm whether plasma ctDNA sequencing panel includes TMB calculation. If not, consider adding pTMB as an exploratory endpoint.
- URGENCY: LOW — hypothesis-generating; no validated threshold yet.

### NDA-09: CMS Subtype Capture Plan
- BASIS: N-001 — CMS4 associated with IO response in REGONIVO (PMID 38336371).
- REQUEST: Confirm whether CMS subtype (CMS1–4) is planned as a biomarker from tumor biopsies.
- URGENCY: LOW — hypothesis-generating; requires biopsy compliance.

### NDA-10: Antigen Characterization Data (Full Multi-Omics Package)
- BASIS: AACR 2024 press release [13] mentions "multi-omics analysis of STC-1010 batches confirmed consistent quality and batch-to-batch reproducibility."
- REQUEST: Full multi-omics data package including:
  (a) Proteomics: antigen catalog per cell line (HCT116, HT29, LoVo)
  (b) Transcriptomics: gene expression changes induced by stress conditions
  (c) Batch-to-batch reproducibility data (n=3 batches per press release)
  (d) Haptenation efficiency data
- URGENCY: LOW for external communication; HIGH for internal scientific positioning.

---

## SUMMARY TABLE

| ID | Topic | Priority | Urgency | Protocol Amendment Required? |
|----|-------|----------|---------|------------------------------|
| NDA-01 | SC-001 antigen overlap (HT29 BRAF V600E) | 1 | HIGH | NO — internal data request |
| NDA-02 | CT051 full poster data | 1 | HIGH | NO — internal data request |
| NDA-03 | Liver met stratification plan | 1 | HIGH | YES — if not currently captured |
| NDA-04 | ctDNA platform + threshold | 2 | MEDIUM | NO — exploratory endpoint |
| NDA-05 | Bevacizumab stratification | 2 | MEDIUM | YES — if not currently stratified |
| NDA-06 | KRAS subtype + SMAD4 capture | 2 | MEDIUM | NO — exploratory biomarker |
| NDA-07 | Biopsy schedule + biomarker panel | 2 | MEDIUM | NO — clarification only |
| NDA-08 | pTMB capture | 3 | LOW | NO — exploratory endpoint |
| NDA-09 | CMS subtype capture | 3 | LOW | NO — exploratory biomarker |
| NDA-10 | Full multi-omics antigen package | 3 | LOW | NO — internal data request |
"""

with open("/workspace/phd_outputs/nda_request_list.md", "w") as f:
    f.write(nda_request_list)

print("Saved nda_request_list.md")
print(f"Total NDA asks: 10 (3 Priority 1, 4 Priority 2, 3 Priority 3)")
print(f"Protocol amendments potentially required: NDA-03 (liver met stratification), NDA-05 (bevacizumab stratification)")


Saved nda_request_list.md
Total NDA asks: 10 (3 Priority 1, 4 Priority 2, 3 Priority 3)
Protocol amendments potentially required: NDA-03 (liver met stratification), NDA-05 (bevacizumab stratification)


Copy all deliverables to /mnt/results/

In [43]:

import shutil, os

# Copy all three deliverables to /mnt/results/
os.makedirs("/mnt/results/phd_evidence_wall", exist_ok=True)

files_to_copy = [
    ("/workspace/phd_outputs/biomarker_timing_matrix.csv", "/mnt/results/phd_evidence_wall/biomarker_timing_matrix.csv"),
    ("/workspace/phd_outputs/contradiction_ledger_v2.md", "/mnt/results/phd_evidence_wall/contradiction_ledger_v2.md"),
    ("/workspace/phd_outputs/nda_request_list.md", "/mnt/results/phd_evidence_wall/nda_request_list.md"),
]

# Also copy master_trial_index.csv if it exists
if os.path.exists("/workspace/phd_outputs/master_trial_index.csv"):
    files_to_copy.append(("/workspace/phd_outputs/master_trial_index.csv", "/mnt/results/phd_evidence_wall/master_trial_index.csv"))

for src, dst in files_to_copy:
    if os.path.exists(src):
        shutil.copy(src, dst)
        size = os.path.getsize(dst)
        print(f"Copied: {os.path.basename(dst)} ({size:,} bytes)")
    else:
        print(f"MISSING: {src}")

print("\nAll deliverables staged to /mnt/results/phd_evidence_wall/")


Copied: biomarker_timing_matrix.csv (21,553 bytes)
Copied: contradiction_ledger_v2.md (14,067 bytes)
Copied: nda_request_list.md (8,128 bytes)
Copied: master_trial_index.csv (25,652 bytes)

All deliverables staged to /mnt/results/phd_evidence_wall/


Build closure audit document — three columns per claim, exact sources, residual uncertainty

In [45]:

# Build the closure audit document — three columns per claim, exact sources, residual uncertainty

closure_audit = """# Upgraded-Source Closure Audit
Generated: 2026-04-30
Scope: Four claims whose status changed from PROVISIONAL/NOT-IN-CANON to CONFIRMED/NEW-FINDING.
Standard: Each entry must attach the exact primary source, archive the key extracted text,
record source tier, and state residual uncertainty explicitly.
Format: Three columns — (1) Claim upgraded, (2) Exact source that cleared it, (3) Residual uncertainty.

---

## AUDIT-01: C-001 — Cell line identity (HCT116, HT29, LoVo)
**Prior status:** PROVISIONAL (company website only)
**New status:** CONFIRMED

### Column 1 — Claim upgraded
STC-1010 is produced from three allogeneic human CRC cell lines: HCT116, HT29, and LoVo.

### Column 2 — Exact sources that cleared it

**Source A — ASCO 2023 Abstract e14661**
- URL: https://ascopubs.org/doi/10.1200/JCO.2023.41.16_suppl.e14661
- Source tier: CONFERENCE ABSTRACT (ASCO Annual Meeting 2023, peer-reviewed abstract selection process)
- Exact extracted text: "We confirmed these proof-of-concept efficacy with the human STC-1010 using
  three CRC human cell lines (HCT116, HT29 and Lovo)."
- Authors: Brenus Pharma authors (Alzeeb et al.)
- Retrieval method: Direct WebFetch of ascopubs.org DOI — full abstract text retrieved successfully
- Archive status: Full abstract text captured in session; not paywalled at abstract level

**Source B — EACR 2025 Abstract EACR25-1044**
- URL: https://storage.unitedwebnetwork.com/files/991/abstract/EACR25-1044.htm
- Source tier: CONFERENCE ABSTRACT (EACR 2025, abstract selection process)
- Exact extracted text: "STC-1010, a therapeutic colorectal cancer (CRC) vaccine, developed by exposing
  HCT116, HT-29, and LoVo cell lines to stress conditions — serum deprivation, irradiation,
  heat shock, and chemotherapy — mimicking standard CRC treatments and relapse conditions."
- Authors: Zbinden M, Richert I, Gamma T, Rudnik M, Chalus L, Pinteur B, Agarkova I,
  Bravetti P, Ligeon L, Alzeeb G (Insphero + Brenus Pharma)
- Retrieval method: Direct WebFetch — full abstract text retrieved successfully
- Archive status: Full abstract text captured in session

**Source C — AACR 2024 Press Release (BioSpace)**
- URL: https://www.biospace.com/aacr-2024-brenus-pharma-unveils-promising-and-robust-pre-clinical-efficacy-extrapolating-human-conditions-with-stc-1010
- Source tier: PRESS RELEASE (company-controlled)
- Exact extracted text: "The study used three human colorectal cancer cell lines: HCT116, HT29, and LoVo,
  to create the STC-1010 vaccine."
- Retrieval method: Direct WebFetch — full press release text retrieved successfully
- Archive status: Full text captured in session

### Column 3 — Residual uncertainty
- **Source A and B are both Brenus-affiliated outputs** (Brenus authors on both). They are
  independent in venue (ASCO vs EACR) but not independent in data origin — both report
  Brenus's own characterization of their own product. There is no independent third-party
  confirmation of cell line identity.
- **No regulatory filing (IND, CTA) or independent laboratory confirmation** of cell line
  identity is in the public record.
- **Consequence:** The claim is confirmed to the standard of "two concordant conference
  abstracts from the sponsor." It is NOT confirmed to the standard of "independently verified
  by a third party." For external scientific assertions, this distinction matters.
- **Recommended language:** "STC-1010 is reported to be produced from HCT116, HT29, and LoVo
  cell lines, as stated in two conference abstracts (ASCO 2023 e14661; EACR 2025 EACR25-1044)."
  Do not write "confirmed" without the source qualifier.

---

## AUDIT-02: C-002 — SC-001 conflict (HT29 BRAF V600E vs BreAK CRC-001 BRAF WT eligibility)
**Prior status:** PROVISIONAL (pending B-001 resolution)
**New status:** CONFIRMED CONFLICT — real inconsistency, unknown clinical consequence

### Column 1 — Claim upgraded
HT29 carries BRAF V600E. BreAK CRC-001 requires BRAF WT at enrollment. This creates a
documented antigen mismatch between one vaccine component and the enrolled patient population.

### Column 2 — Exact sources that cleared it

**Source A — HT29 BRAF V600E status**
- This is established cell line biology, not a new finding. HT29 BRAF V600E is documented in:
  ATCC product sheet (HTB-38), CCLE (Broad Institute), COSMIC Cell Lines database, and
  multiple published papers. It is not in dispute.
- Source tier: ESTABLISHED CELL LINE BIOLOGY — no citation needed; universally accepted.

**Source B — BreAK CRC-001 BRAF WT eligibility**
- Source: NCT06934538 registry, Inclusion Criterion #4
- Exact extracted text (from prior session registry retrieval): "BRAF wild-type (WT) status
  confirmed by local laboratory testing" [paraphrased from registry — exact wording requires
  NDA data package for verbatim text; registry summary confirms BRAF WT requirement]
- Source tier: PRIMARY REGISTRY (ClinicalTrials.gov API v2)
- Retrieval method: ClinicalTrials.gov API v2 query in prior session

**Source C — Cell line identity confirmation (closes the logical chain)**
- ASCO 2023 e14661 and EACR 2025 EACR25-1044 (see AUDIT-01) confirm HT29 is a vaccine component.
- The conflict is therefore: HT29 (BRAF V600E, vaccine component) + BRAF WT eligibility
  (registry) = antigen mismatch.

### Column 3 — Residual uncertainty
- **The conflict is real. The clinical consequence is unknown.**
- The haptenization mechanism is designed to expose shared tumor antigens (stress-response
  proteins, calreticulin, HMGB1, HSPs) that are not mutation-specific. If the dominant
  immunogenic antigens in STC-1010 are stress-response antigens rather than BRAF V600E-specific
  neoantigens, the mismatch may be clinically irrelevant.
- **No public data exists** comparing the antigen profile of HT29-derived STC-1010 components
  vs HCT116/LoVo-derived components, or vs BRAF WT patient tumor proteomes.
- The AACR 2024 press release mentions "multi-omics analysis of STC-1010 batches" — this data
  package exists internally at Brenus but is not in the public record.
- **Recommended language:** "STC-1010 contains antigens derived from HT29 (BRAF V600E), while
  BreAK CRC-001 enrolls only BRAF WT patients. The clinical significance of this antigen
  mismatch is unknown and requires sponsor disclosure of antigen overlap data (NDA-01)."
- **Do not write:** "SC-001 is a definitive efficacy concern." It is an open scientific
  question, not a proven problem.

---

## AUDIT-03: C-003 — AtezoTRIBE liver-met nuance
**Prior status:** CANON ("liver mets = IO non-permissive")
**New status:** NUANCED — prior canon overstated; correct framing is more conditional

### Column 1 — Claim upgraded
Liver metastases alone do not predict resistance to atezolizumab in pMMR mCRC. Immunoscore-IC
retains predictive impact even in the liver-met subgroup. The correct framing is
"liver mets + Immunoscore-IC-low = non-permissive," not "liver mets alone = non-permissive."

### Column 2 — Exact source that cleared it

**Source — PMID 39908699**
- Full citation: Antoniotti C et al. "Liver metastases do not predict resistance to the addition
  of atezolizumab to first-line FOLFOXIRI plus bevacizumab in proficient MMR metastatic
  colorectal cancer: a secondary analysis of the AtezoTRIBE study." ESMO Open. 2025 Feb 4;
  10(2):104135. doi: 10.1016/j.esmoop.2025.104135
- Source tier: PEER-REVIEWED (ESMO Open, open access, published 2025-02-04)
- URL: https://pubmed.ncbi.nlm.nih.gov/39908699/
- Exact extracted text from abstract:
  "Out of 202 pMMR patients, 151 (75%) had LMs. No differences in immune-related features
  were observed according to the presence or not of LMs, except for a lower prevalence of
  tumour-infiltrating lymphocytes-high tumours in the LMs group (33% versus 52%, P = 0.03).
  Worse outcomes were observed among patients with LMs [PFS, P = 0.002; OS, P = 0.011],
  also in multivariable models. The effect of adding atezolizumab to FOLFOXIRI/bevacizumab
  was independent from LMs in terms of PFS (Pint = 0.990) and OS (Pint = 0.800). Among
  patients with pMMR mCRC and LMs, those with immunoscore-IC-high but not those with
  immunoscore-IC-low tumours achieved benefit from atezolizumab, though in the absence of
  a statistically significant interaction effect (Pint for PFS and OS = 0.166 and 0.473,
  respectively)."
  Conclusion: "LMs are associated with poor prognosis in pMMR mCRC and do not predict
  resistance to the addition of atezolizumab to FOLFOXIRI/bevacizumab. Immunoscore-IC seems
  to retain its predictive impact also among patients with LMs."
- Retrieval method: Direct WebFetch of PubMed abstract — full abstract text retrieved
- Archive status: Full abstract text captured in session

### Column 3 — Residual uncertainty
- **This is a secondary analysis of a Phase 2 trial (n=202 pMMR), not a pre-specified
  primary endpoint.** The interaction tests (Pint = 0.990 for PFS, 0.800 for OS) are
  exploratory. A non-significant interaction test does NOT prove that liver mets are
  irrelevant — it means the study was underpowered to detect a differential effect.
- **The Immunoscore-IC interaction in the liver-met subgroup is also non-significant**
  (Pint = 0.166 for PFS, 0.473 for OS). The authors' conclusion that "Immunoscore-IC seems
  to retain its predictive impact" is a directional observation, not a statistically
  validated finding.
- **The BOT/BAL and RIN no-liver-met signals are unchanged** and remain the strongest
  clinical evidence for liver-met stratification in MSS mCRC IO. AtezoTRIBE 2025 does not
  contradict those findings — it addresses a different drug (atezolizumab), a different
  backbone (FOLFOXIRI), and a different patient population (pMMR, not MSS-restricted).
- **Critical distinction:** AtezoTRIBE enrolled pMMR patients (not MSS-restricted). The
  BOT/BAL and RIN liver-met findings are in MSS-restricted populations. These are not
  directly comparable.
- **Recommended language:** "A secondary analysis of AtezoTRIBE (PMID 39908699) found that
  liver metastases were prognostic but not predictive of atezolizumab resistance in pMMR
  mCRC (Pint PFS=0.990, OS=0.800), with Immunoscore-IC retaining directional predictive
  impact in the liver-met subgroup. These are exploratory findings from a Phase 2 secondary
  analysis and do not supersede the validated no-liver-met enrichment strategy demonstrated
  in MSS-restricted populations (BOT/BAL, RIN)."
- **Do not use AtezoTRIBE 2025 to argue that liver-met patients should be enrolled in
  BreAK CRC-001 without stratification.** The populations are not equivalent.

---

## AUDIT-04: N-002 — pTMB ≥28 mut/Mb as predictive biomarker
**Prior status:** NOT IN CANON
**New status:** NEW FINDING — peer-reviewed, but with critical caveats on threshold derivation

### Column 1 — Claim upgraded
Plasma TMB (pTMB) ≥28 mut/Mb is predictive of benefit from durvalumab + tremelimumab in
MSS mCRC (HR 0.34, 95%CI 0.13–0.85, p=0.022, Pint=0.070). Tissue TMB ≥10 mut/Mb is NOT
predictive (HR 0.71, p=0.47, Pint=0.62).

### Column 2 — Exact source that cleared it

**Source — PMID 38727700**
- Full citation: Loree JM et al. "Plasma versus Tissue Tumor Mutational Burden as Biomarkers
  of Durvalumab plus Tremelimumab Response in Patients with Metastatic Colorectal Cancer in
  the CO.26 Trial." Clin Cancer Res. 2024 Aug 1;30(15):3189–3199.
  doi: 10.1158/1078-0432.CCR-24-0268
- Source tier: PEER-REVIEWED (Clinical Cancer Research, AACR journal, open access)
- URL: https://pmc.ncbi.nlm.nih.gov/articles/PMC11292199/
- Trial: CO.26 (NCT02870920) — NOT MEDITREME. This is the Canadian Cancer Trials Group
  randomized Phase 2 trial of durvalumab + tremelimumab vs BSC in refractory mCRC (n=180).
  MEDITREME is a separate trial (NCT03202758, Nature Medicine 2023). The pTMB finding is
  from CO.26, not MEDITREME.
- Exact extracted text:
  "pTMB ≥28 mutations/Mb was predictive of benefit from D+T [HR, 0.34 (95% CI, 0.13–0.85);
  P=0.022], as was clonal pTMB ≥10.6 mutations/Mb [HR, 0.10 (95% CI, 0.014–0.79); P=0.029]
  and subclonal pTMB ≥25.9/Mb [HR, 0.20 (95% CI, 0.061–0.69); P=0.010]."
  "Tissue TMB ≥10 mutations/Mb was not predictive of benefit from D+T compared with BSC in
  microsatellite stable (MSS) metastatic CRC [HR, 0.71 (95% CI, 0.28–1.80); P=0.47]."
  "No tissue TMB threshold could identify a high TMB group that benefited from ICI."
- Platform: GuardantOMNI (500-gene panel, Guardant Health) — platform-specific threshold
- Threshold derivation: "Cases were classified as TMB high by plasma if TMB was ≥28
  mutations/Mb based on our prior work and lack of other available cut points suggestive of
  the appropriate threshold." — i.e., DATA-DRIVEN from prior exploratory analysis, NOT
  pre-specified in the CO.26 protocol.
- Retrieval method: Direct WebFetch of PMC full text — complete paper retrieved successfully
- Archive status: Full paper text captured in session

### Column 3 — Residual uncertainty — FOUR CRITICAL CAVEATS
1. **TRIAL ATTRIBUTION ERROR IN PRIOR CANON:** The pTMB finding was attributed to MEDITREME
   in the biomarker matrix. This is WRONG. The pTMB ≥28 mut/Mb finding is from CO.26
   (PMID 38727700), not MEDITREME. MEDITREME (PMID 37563240) is a separate trial with
   different biomarker findings. The biomarker_timing_matrix.csv row for "MEDITREME / CO.26"
   conflated these two trials. This must be corrected in the matrix.

2. **THRESHOLD IS DATA-DRIVEN, NOT PRE-SPECIFIED.** The ≥28 mut/Mb threshold was derived
   from a minimal P-value approach in a prior exploratory analysis and carried forward. The
   interaction P-value is 0.070 — borderline, not significant at conventional alpha. This
   threshold requires prospective validation before clinical use.

3. **PLATFORM-SPECIFIC.** The threshold applies to GuardantOMNI pTMB specifically. The
   authors explicitly state: "Our threshold of pTMB may not be directly applicable to pTMB
   derived from other assays." Different ctDNA platforms will produce different pTMB values
   for the same sample.

4. **POPULATION MISMATCH.** CO.26 enrolled refractory mCRC (≥2L), not 1L. BreAK CRC-001
   is 1L. Prior therapy (especially anti-EGFR exposure) significantly increases pTMB in
   CO.26 (P=2.44×10⁻⁶). In a 1L population with less prior therapy, pTMB distributions
   will be lower and the ≥28 mut/Mb threshold may select a different (smaller) fraction
   of patients.

- **Recommended language:** "A secondary analysis of CO.26 (PMID 38727700) found that
  pTMB ≥28 mut/Mb (GuardantOMNI) was predictive of benefit from durvalumab + tremelimumab
  in refractory MSS mCRC (HR 0.34, Pint=0.070). The threshold was data-driven, is
  platform-specific, and requires prospective validation. It is not directly applicable to
  a 1L population or to other ctDNA platforms."
- **Do not write:** "pTMB ≥28 mut/Mb is a validated predictive biomarker." It is a
  hypothesis-generating finding from a single exploratory analysis.

---

## SUMMARY TABLE

| Audit ID | Claim | Old status | New status | Source tier | Trial attribution correct? | Residual uncertainty level |
|----------|-------|-----------|-----------|-------------|---------------------------|---------------------------|
| AUDIT-01 | Cell lines HCT116/HT29/LoVo | PROVISIONAL | CONFIRMED (sponsor-reported) | 2× CONFERENCE ABSTRACT | YES | LOW — two concordant sources; no independent third-party verification |
| AUDIT-02 | SC-001 HT29 BRAF V600E conflict | PROVISIONAL | CONFIRMED CONFLICT | ESTABLISHED BIOLOGY + REGISTRY | YES | MEDIUM — conflict is real; clinical consequence unknown |
| AUDIT-03 | Liver mets nuance (AtezoTRIBE) | CANON (overstated) | NUANCED | PEER-REVIEWED (ESMO Open 2025) | YES | HIGH — exploratory secondary analysis; pMMR not MSS-restricted; non-significant interactions |
| AUDIT-04 | pTMB ≥28 mut/Mb (CO.26) | NOT IN CANON | NEW FINDING (provisional) | PEER-REVIEWED (Clin Cancer Res 2024) | PARTIALLY — prior canon misattributed to MEDITREME; correct trial is CO.26 | HIGH — data-driven threshold, Pint=0.070, platform-specific, 2L+ population |

---

## CORRECTIONS REQUIRED IN BIOMARKER_TIMING_MATRIX.CSV

### CORRECTION-01: Trial attribution for pTMB finding
- AFFECTED ROW: "MEDITREME / CO.26 (MSS subgroup)"
- ERROR: pTMB ≥28 mut/Mb finding attributed to MEDITREME in the key_biomarker_finding field
- CORRECT: pTMB ≥28 mut/Mb is from CO.26 (PMID 38727700), not MEDITREME
- MEDITREME (PMID 37563240) is a separate trial with different biomarker findings
  (immune signature, EMT signature — not pTMB)
- ACTION: Split into two separate rows: "CO.26" and "MEDITREME" with correct biomarker
  findings per trial

### CORRECTION-02: Pint values for AtezoTRIBE liver-met interaction
- AFFECTED ROW: "AtezoTRIBE (pMMR subgroup)"
- PRIOR ENTRY: "Pint PFS=0.990, OS=0.800" (for liver met vs no liver met interaction with
  atezolizumab arm) — CORRECT
- ADDITIONAL DATA NOW CONFIRMED: Immunoscore-IC interaction within liver-met subgroup:
  Pint PFS=0.166, OS=0.473 — these were stated in prior canon but now confirmed from
  PubMed abstract text
- ACTION: Add Immunoscore-IC Pint values to the immunoscore_capture field

### CORRECTION-03: CO.26 pTMB platform specification
- AFFECTED ROW: "MEDITREME / CO.26 (MSS subgroup)"
- PRIOR ENTRY: "Plasma ctDNA sequencing (platform not specified in abstract)"
- CORRECT: GuardantOMNI (500-gene panel, Guardant Health) — confirmed from full paper text
- ACTION: Update ctdna_platform field to "GuardantOMNI (500-gene, Guardant Health)"
"""

import os
os.makedirs("/workspace/phd_outputs", exist_ok=True)
with open("/workspace/phd_outputs/upgraded_source_closure_audit.md", "w") as f:
    f.write(closure_audit)

print("Saved upgraded_source_closure_audit.md")
print(f"Length: {len(closure_audit):,} characters")
print("\nCorrections required in biomarker_timing_matrix.csv:")
print("  CORRECTION-01: pTMB finding misattributed to MEDITREME — correct trial is CO.26")
print("  CORRECTION-02: Add Immunoscore-IC Pint values (0.166/0.473) to AtezoTRIBE row")
print("  CORRECTION-03: Update CO.26 ctDNA platform to GuardantOMNI")


Saved upgraded_source_closure_audit.md
Length: 17,463 characters

Corrections required in biomarker_timing_matrix.csv:
  CORRECTION-01: pTMB finding misattributed to MEDITREME — correct trial is CO.26
  CORRECTION-02: Add Immunoscore-IC Pint values (0.166/0.473) to AtezoTRIBE row
  CORRECTION-03: Update CO.26 ctDNA platform to GuardantOMNI


Verify current matrix state before applying corrections

In [47]:

import pandas as pd

df = pd.read_csv("/workspace/phd_outputs/biomarker_timing_matrix.csv")

# Verify current state before corrections
print("=== PRE-CORRECTION VERIFICATION ===")
print(f"Total rows: {len(df)}")
print(f"\nRow containing MEDITREME/CO.26:")
meditreme_row = df[df['trial'].str.contains('MEDITREME', na=False)]
print(meditreme_row[['trial', 'nct', 'ctdna_platform', 'key_biomarker_finding']].to_string())

print(f"\nRow containing AtezoTRIBE:")
atezo_row = df[df['trial'].str.contains('AtezoTRIBE', na=False)]
print(atezo_row[['trial', 'immunoscore_capture', 'key_biomarker_finding']].to_string())


=== PRE-CORRECTION VERIFICATION ===
Total rows: 16

Row containing MEDITREME/CO.26:
                               trial          nct                                                ctdna_platform                                                                                                                                               key_biomarker_finding
12  MEDITREME / CO.26 (MSS subgroup)  NCT03202758  Plasma ctDNA sequencing (platform not specified in abstract)  pTMB (plasma) is predictive; tissue TMB is NOT. Neoantigen-specific T-cell responses associated with better PFS. High immune signature + low EMT = better outcome.

Row containing AtezoTRIBE:
                         trial                                                                               immunoscore_capture                                                                                                                                                                                                                

Apply three corrections to biomarker matrix — split CO.26/MEDITREME, fix Pint values, fix platform

In [49]:

import pandas as pd, shutil

df = pd.read_csv("/workspace/phd_outputs/biomarker_timing_matrix.csv")

# ── CORRECTION-01: Split MEDITREME/CO.26 into two rows ──────────────────────
# First, update the existing row to be CO.26-only with correct pTMB attribution
co26_idx = df[df['trial'].str.contains('MEDITREME', na=False)].index[0]

# CO.26 row — pTMB finding belongs here
df.at[co26_idx, 'trial'] = 'CO.26 (CCTG)'
df.at[co26_idx, 'nct'] = 'NCT02870920'
df.at[co26_idx, 'comparator_class'] = 'SUBGROUP-RICH EVIDENCE ANCHOR'
df.at[co26_idx, 'evidence_quality'] = 'PEER-REVIEWED'
df.at[co26_idx, 'source_pmid'] = '38727700 (pTMB paper); 37563240 (main CO.26 paper JAMA Oncol 2020)'
df.at[co26_idx, 'line'] = '≥2L refractory'
df.at[co26_idx, 'mss_restriction'] = 'NO — all-comer mCRC; 98% MSS by plasma sequencing'
df.at[co26_idx, 'ctdna_use'] = 'YES — pTMB as EXPLORATORY predictive biomarker (post-hoc, data-driven threshold)'
df.at[co26_idx, 'ctdna_platform'] = 'GuardantOMNI (500-gene panel, Guardant Health) — confirmed from PMID 38727700 full text'
df.at[co26_idx, 'ctdna_timepoints'] = 'Baseline only'
df.at[co26_idx, 'ctdna_threshold'] = 'pTMB ≥28 mut/Mb (GuardantOMNI-specific; data-driven via minimal P-value approach from prior work; NOT pre-specified in CO.26 protocol; Pint=0.070 — borderline, not significant at conventional alpha)'
df.at[co26_idx, 'bevacizumab_structure'] = 'NOT USED — durvalumab + tremelimumab vs BSC (no chemotherapy backbone)'
df.at[co26_idx, 'bevacizumab_timing'] = 'NOT APPLICABLE'
df.at[co26_idx, 'enrichment_design'] = 'POST-HOC — pTMB subgroup identified via data-driven threshold from prior exploratory analysis'
df.at[co26_idx, 'enrichment_biomarker'] = 'pTMB ≥28 mut/Mb (GuardantOMNI; data-driven; NOT pre-specified)'
df.at[co26_idx, 'primary_endpoint'] = 'OS (co-primary with PFS)'
df.at[co26_idx, 'primary_result'] = 'Final OS: HR 0.68 (95%CI 0.49–0.93), p=0.017 (all-comer). pTMB ≥28 mut/Mb subgroup: HR 0.34 (95%CI 0.13–0.85), p=0.022, Pint=0.070. Tissue TMB ≥10 mut/Mb: HR 0.71 (95%CI 0.28–1.80), p=0.47, Pint=0.62 — NOT predictive.'
df.at[co26_idx, 'key_biomarker_finding'] = (
    'TRIAL ATTRIBUTION CORRECTED (prior canon misattributed this to MEDITREME — error). '
    'pTMB ≥28 mut/Mb (GuardantOMNI) is predictive of D+T benefit in refractory MSS mCRC '
    '(HR 0.34, Pint=0.070). Tissue TMB NOT predictive. FOUR CAVEATS: (1) threshold is '
    'data-driven not pre-specified; (2) Pint=0.070 is borderline; (3) GuardantOMNI-specific '
    'threshold not transferable to other platforms; (4) 2L+ population — not directly '
    'applicable to 1L BreAK CRC-001 where prior therapy-driven pTMB elevation is absent.'
)
df.at[co26_idx, 'canon_status'] = 'PEER-REVIEWED — Clin Cancer Res 2024 (PMID 38727700). TRIAL ATTRIBUTION CORRECTED from MEDITREME to CO.26.'
df.at[co26_idx, 'dth_immune_readouts'] = 'NOT REPORTED'
df.at[co26_idx, 'dth_platform'] = 'NOT FOUND'
df.at[co26_idx, 'dth_timepoints'] = 'NOT FOUND'
df.at[co26_idx, 'biopsy_schedule'] = 'REQUIRED — archival FFPE tissue for WES (tissue TMB); plasma for ctDNA (GuardantOMNI)'
df.at[co26_idx, 'kras_subtype_capture'] = 'NOT FOUND as subtype'
df.at[co26_idx, 'smad4_capture'] = 'NOT FOUND'
df.at[co26_idx, 'immunoscore_capture'] = 'NOT FOUND'
df.at[co26_idx, 'failure_mode_primary'] = 'S — unselected enrollment diluted pTMB-high signal'
df.at[co26_idx, 'failure_mode_contributing'] = 'NOT APPLICABLE'

# Now add MEDITREME as a new separate row
meditreme_new = {
    'trial': 'MEDITREME',
    'nct': 'NCT03202758',
    'comparator_class': 'SUBGROUP-RICH EVIDENCE ANCHOR',
    'evidence_quality': 'PEER-REVIEWED',
    'source_pmid': '37563240 (main); NOTE: pTMB finding is from CO.26 PMID 38727700, NOT this trial',
    'line': '1L',
    'mss_restriction': 'YES — MSS subgroup (RAS-mutant mCRC, MSS subset analyzed)',
    'liver_met_handling': 'NOT STRATIFIED — no liver met restriction or stratification',
    'liver_met_source': 'Registry/publication: no liver met criterion',
    'ctdna_use': 'NOT USED as primary endpoint — no pTMB analysis in MEDITREME',
    'ctdna_platform': 'NOT FOUND in MEDITREME',
    'ctdna_timepoints': 'NOT FOUND',
    'ctdna_threshold': 'NOT FOUND — pTMB ≥28 mut/Mb is from CO.26, NOT MEDITREME',
    'biopsy_schedule': 'REQUIRED — tumor tissue for immune signature and EMT signature analysis',
    'dth_immune_readouts': 'YES — neoantigen and NY-ESO1/TERT blood T-cell responses measured; associated with better PFS',
    'dth_platform': 'Blood T-cell response assay (platform not specified in abstract)',
    'dth_timepoints': 'On-treatment (timing not specified)',
    'bevacizumab_structure': 'NOT USED — durvalumab + tremelimumab + mFOLFOX6 (no bevacizumab)',
    'bevacizumab_timing': 'NOT APPLICABLE',
    'enrichment_design': 'POST-HOC — immune signature and EMT signature subgroup analyses',
    'enrichment_biomarker': 'High immune signature; low EMT signature (not pTMB)',
    'kras_subtype_capture': 'NOT FOUND',
    'smad4_capture': 'NOT FOUND',
    'immunoscore_capture': 'NOT FOUND — immune signature used but not Immunoscore-IC specifically',
    'primary_endpoint': 'Phase 2: 3-month PFS rate in MSS patients',
    'primary_result': '3-month PFS 90.7% (95%CI 79.2–96%) — primary endpoint MET. ORR 64.5%, mPFS 8.2 months. High immune + low EMT signature associated with better outcomes.',
    'failure_mode_primary': 'NOT A FAILURE — single-arm Phase 1b/2, primary endpoint met. But no randomized control; 3-month PFS is a weak endpoint.',
    'failure_mode_contributing': 'A — single-arm; 3-month PFS endpoint too early to assess IO contribution',
    'key_biomarker_finding': (
        'TRIAL ATTRIBUTION NOTE: pTMB ≥28 mut/Mb finding is from CO.26 (PMID 38727700), NOT MEDITREME. '
        'MEDITREME biomarker findings: neoantigen-specific T-cell responses associated with better PFS; '
        'high immune signature + low EMT signature = better outcome. These are distinct from the CO.26 pTMB finding.'
    ),
    'canon_status': 'PEER-REVIEWED — Nature Medicine 2023 (PMID 37563240). NOTE: prior canon conflated MEDITREME and CO.26 biomarker findings. Now separated.',
}

df = pd.concat([df, pd.DataFrame([meditreme_new])], ignore_index=True)

# ── CORRECTION-02: Update AtezoTRIBE Immunoscore-IC Pint values ─────────────
atezo_idx = df[df['trial'].str.contains('AtezoTRIBE', na=False)].index[0]

# Correct the Pint value — prior entry had 0.089 which was from a different AtezoTRIBE paper
# Confirmed from PMID 39908699 abstract: Pint PFS=0.990, OS=0.800 (liver met vs no liver met)
# Immunoscore-IC within liver-met subgroup: Pint PFS=0.166, OS=0.473
df.at[atezo_idx, 'immunoscore_capture'] = (
    'YES — Immunoscore-IC measured. '
    'Liver met vs no-liver-met interaction with atezolizumab: Pint PFS=0.990, OS=0.800 (non-significant). '
    'Immunoscore-IC within liver-met subgroup: Pint PFS=0.166, OS=0.473 (non-significant). '
    'All interaction tests are EXPLORATORY from a secondary analysis. '
    'Source: PMID 39908699 (ESMO Open 2025-02-04) — confirmed from PubMed abstract.'
)
df.at[atezo_idx, 'key_biomarker_finding'] = (
    'CANON NUANCE (CORRECTED): Liver mets are prognostic (worse PFS p=0.002, OS p=0.011) but '
    'NOT predictive of atezolizumab resistance in pMMR mCRC (Pint PFS=0.990, OS=0.800). '
    'Immunoscore-IC retains directional predictive impact in liver-met subgroup but interaction '
    'is non-significant (Pint PFS=0.166, OS=0.473). FOUR CAVEATS: (1) secondary analysis of '
    'Phase 2 trial; (2) pMMR not MSS-restricted — not directly comparable to BOT/BAL or RIN; '
    '(3) all interaction tests exploratory; (4) non-significant Pint does not prove equivalence. '
    'BOT/BAL and RIN no-liver-met signals in MSS-restricted populations remain the strongest '
    'validated patient selection evidence and are NOT superseded by this finding.'
)
df.at[atezo_idx, 'source_pmid'] = '38865678 (4yr update); 39908699 (liver met secondary analysis, ESMO Open 2025-02-04, confirmed from PubMed abstract)'

print("=== POST-CORRECTION VERIFICATION ===")
print(f"Total rows: {len(df)} (was 16, now 17 — MEDITREME split from CO.26)")

print(f"\nCO.26 row:")
co26_check = df[df['trial'] == 'CO.26 (CCTG)']
print(co26_check[['trial', 'nct', 'ctdna_platform', 'ctdna_threshold']].to_string())

print(f"\nMEDITREME row:")
med_check = df[df['trial'] == 'MEDITREME']
print(med_check[['trial', 'nct', 'ctdna_use', 'key_biomarker_finding']].to_string())

print(f"\nAtezoTRIBE row (Immunoscore-IC field):")
atezo_check = df[df['trial'].str.contains('AtezoTRIBE', na=False)]
print(atezo_check[['trial', 'immunoscore_capture']].to_string())

# Save corrected matrix
df.to_csv("/workspace/phd_outputs/biomarker_timing_matrix.csv", index=False)
print(f"\nSaved corrected biomarker_timing_matrix.csv — {len(df)} rows x {len(df.columns)} columns")


=== POST-CORRECTION VERIFICATION ===
Total rows: 17 (was 16, now 17 — MEDITREME split from CO.26)

CO.26 row:
           trial          nct                                                                           ctdna_platform                                                                                                                                                                                         ctdna_threshold
12  CO.26 (CCTG)  NCT02870920  GuardantOMNI (500-gene panel, Guardant Health) — confirmed from PMID 38727700 full text  pTMB ≥28 mut/Mb (GuardantOMNI-specific; data-driven via minimal P-value approach from prior work; NOT pre-specified in CO.26 protocol; Pint=0.070 — borderline, not significant at conventional alpha)

MEDITREME row:
        trial          nct                                                     ctdna_use                                                                                                                                                    

Copy corrected matrix and closure audit to /mnt/results/

In [51]:

import shutil, os

# Copy corrected matrix and new closure audit to results
shutil.copy("/workspace/phd_outputs/biomarker_timing_matrix.csv",
            "/mnt/results/phd_evidence_wall/biomarker_timing_matrix.csv")
shutil.copy("/workspace/phd_outputs/upgraded_source_closure_audit.md",
            "/mnt/results/phd_evidence_wall/upgraded_source_closure_audit.md")

for f in ["biomarker_timing_matrix.csv", "upgraded_source_closure_audit.md"]:
    size = os.path.getsize(f"/mnt/results/phd_evidence_wall/{f}")
    print(f"Staged: {f} ({size:,} bytes)")


Staged: biomarker_timing_matrix.csv (24,542 bytes)
Staged: upgraded_source_closure_audit.md (17,625 bytes)


Build complete 8D-readiness audit — five sections, redline adjudication table

In [53]:

# Build the 8D-readiness audit as a structured markdown document
# This is a clean-room adjudication — no narrative repair, only classification and redline

audit_8d = """# 8D-Readiness Audit — BreAK CRC-001 Package
Generated: 2026-04-30
Auditor: Biomni / CrisPRO
Scope: Design Intelligence Brief (brenus_outreach_brief.md) + all prior canon
Verified ledger: contradiction_ledger_v2.md + upgraded_source_closure_audit.md
Purpose: Prepare a clean substrate for 8D vector decoding. Classify every claim.
8D must not use BLOCKED claims. PROVISIONAL claims require caveat tags in vector output.

---

## SECTION 1 — VERIFIED CLAIMS SAFE FOR 8D USE

These claims are sourced from peer-reviewed publications or primary registry records,
have been confirmed against the verified ledger, and carry no outstanding contradictions.
8D may use these without caveat tags.

| ID | Claim text | Confidence | Source type | Source ID | Why it matters for 8D | 8D can proceed without sponsor data? |
|----|-----------|-----------|------------|-----------|----------------------|--------------------------------------|
| V-001 | CheckMate 9X8: HR 0.81 (95%CI 0.53–1.23), p=0.30; mPFS 11.86 vs 11.93 months; Phase 3 not triggered | HIGH | PEER-REVIEWED | PMID 38485190 | Null benchmark for PD-1 + mFOLFOX6 + bev in 1L MSS mCRC; sets the floor that STC-1010 must clear | YES |
| V-002 | CheckMate 9X8: n=196, 1L, all-comer mCRC (95% MSS/pMMR), NCT03414983 | HIGH | PEER-REVIEWED + REGISTRY | PMID 38485190; ClinicalTrials.gov | Trial identity and population for comparator positioning | YES |
| V-003 | QUILT-2.004: HR 1.061 (95%CI 0.380–2.966), p=0.91; mPFS 8.8 vs 10.1 months | HIGH | PEER-REVIEWED | PMID 35274710 | Primary negative benchmark; identical backbone to BreAK CRC-001 | YES |
| V-004 | QUILT-2.004: multifunctional CD4+/CD8+ T-cell responses in 8/11 IO patients vs 1/8 SOC (p=0.020) | HIGH | PEER-REVIEWED | PMID 35274710 | Confirms priming-without-trafficking failure mode; directly relevant to STC-1010 mechanism claim | YES |
| V-005 | QUILT-2.004: n=30, 1L, terminated early; NCT03050814 | HIGH | PEER-REVIEWED + REGISTRY | PMID 35274710 | Trial identity; underpowered caveat | YES |
| V-006 | BOT/BAL Phase 1: ORR 17% (95%CI 10–26%), DCR 61%, n=101 response-evaluable MSS mCRC (no liver mets); NCT03860272 | HIGH | PEER-REVIEWED | PMID 38871975 | Strongest validated IO signal in MSS mCRC; liver-met exclusion is the defining design feature | YES |
| V-007 | BOT/BAL Phase 1: active liver metastases explicitly excluded from MSS mCRC cohort (eligibility criterion) | HIGH | PEER-REVIEWED | PMID 38871975 | Source of the no-liver-met enrichment hypothesis | YES |
| V-008 | BOT/BAL Phase 2: cORR 19% (n=62, BOT 75mg+BAL); Phase 3 BATTMAN initiated | MEDIUM | CONFERENCE DATA (ASCO GI 2025) — not yet peer-reviewed | Conference presentation | Confirms Phase 1 signal at Phase 2 scale; use with caveat that not yet peer-reviewed | YES — with caveat: not yet peer-reviewed |
| V-009 | RIN NLM subgroup: ORR 36.4%, mPFS 5.0 months, mOS 27.5 months, 3-year PFS 19.3%; n=22; NCT04362839 | MEDIUM | PEER-REVIEWED | PMID 39504677 | Highest validated IO ORR in MSS mCRC; confirms liver-met stratification; n=22 is small | YES — with caveat: n=22 |
| V-010 | RIN NLM: three patients disease-free >18 months after treatment completion | MEDIUM | PEER-REVIEWED | PMID 39504677 | Durable remissions possible in MSS mCRC with IO; supports OS endpoint rationale | YES — with caveat: n=3 |
| V-011 | AtezoTRIBE liver-met secondary analysis: liver mets prognostic (PFS p=0.002, OS p=0.011) but NOT predictive of atezolizumab resistance (Pint PFS=0.990, OS=0.800) in pMMR mCRC (n=202, 151 with liver mets) | MEDIUM | PEER-REVIEWED | PMID 39908699 (ESMO Open 2025-02-04) | Nuances liver-met framing; Pint values are confirmed from PubMed abstract | YES — with mandatory caveat: exploratory secondary analysis; pMMR not MSS-restricted |
| V-012 | AtezoTRIBE: Immunoscore-IC interaction within liver-met subgroup: Pint PFS=0.166, OS=0.473 (non-significant) | MEDIUM | PEER-REVIEWED | PMID 39908699 | Directional finding only; does not validate Immunoscore-IC as a stratification variable | YES — with mandatory caveat: non-significant, exploratory |
| V-013 | CO.26 pTMB: pTMB ≥28 mut/Mb (GuardantOMNI) predictive of D+T benefit in refractory MSS mCRC; HR 0.34 (95%CI 0.13–0.85), p=0.022, Pint=0.070 | MEDIUM | PEER-REVIEWED | PMID 38727700 (Clin Cancer Res 2024) | Strongest available plasma biomarker hypothesis for MSS mCRC IO; Pint=0.070 is borderline | YES — with mandatory caveat: data-driven threshold, GuardantOMNI-specific, 2L+ population, Pint=0.070 |
| V-014 | CO.26 pTMB: tissue TMB ≥10 mut/Mb NOT predictive (HR 0.71, p=0.47, Pint=0.62) | HIGH | PEER-REVIEWED | PMID 38727700 | Closes the tissue TMB hypothesis; 8D should not use tissue TMB as a predictive biomarker claim | YES |
| V-015 | CO.26 pTMB: threshold is data-driven (minimal P-value approach), NOT pre-specified in CO.26 protocol; GuardantOMNI-specific | HIGH | PEER-REVIEWED | PMID 38727700 (full text) | Critical caveat for any pTMB claim; threshold requires prospective validation | YES |
| V-016 | NCT06934538: biopsy is CONDITIONAL ("only if not representing an unacceptable clinical risk and/or if technically feasible") — Inclusion Criterion #6 | HIGH | PRIMARY REGISTRY | ClinicalTrials.gov API v2 | Limits biomarker data quality; prior canon stated "biopsy required" — that was wrong | YES |
| V-017 | NCT06934538: bevacizumab is OPTIONAL ("mFOLFOX6 with or without bevacizumab") | HIGH | PRIMARY REGISTRY | ClinicalTrials.gov API v2 | Creates immunological heterogeneity; not captured as stratification variable | YES |
| V-018 | NCT06934538: liver metastasis status is NOT an eligibility criterion, stratification variable, or pre-specified covariate | HIGH | PRIMARY REGISTRY | ClinicalTrials.gov API v2 | Confirmed gap; symptomatic ascites excluded but liver met burden not addressed | YES |
| V-019 | NCT06934538: Phase IIa primary endpoint is PFS rate at 12 months (Arm 2A-1, MSS mCRC) | HIGH | PRIMARY REGISTRY | ClinicalTrials.gov API v2 | Defines the interpretability standard for Phase IIa | YES |
| V-020 | NCT06934538: exploratory biomarker timepoints listed as T0, 1m, 2m, 6m, 12m, Tprogression | HIGH | PRIMARY REGISTRY | ClinicalTrials.gov API v2 | ctDNA not explicitly named; "molecular biomarkers" is the registry language | YES |
| V-021 | HT29 carries BRAF V600E — established cell line biology (ATCC HTB-38, CCLE, COSMIC) | HIGH | ESTABLISHED CELL LINE BIOLOGY | ATCC/CCLE/COSMIC | Foundational fact for SC-001 conflict; not in dispute | YES |
| V-022 | NCT06934538 Inclusion Criterion #4 requires BRAF wild-type status | HIGH | PRIMARY REGISTRY | ClinicalTrials.gov API v2 | Closes the SC-001 logical chain with V-021 | YES |
| V-023 | EACR 2025 EACR25-1044: haptenated epitopes detected on primed mDCs; CD8+ T cells induced apoptosis in HCT116, HT29, SW620 vs controls | MEDIUM | CONFERENCE ABSTRACT (sponsor-affiliated) | EACR25-1044 | First public mechanistic confirmation of haptenation at antigen presentation level | YES — with caveat: sponsor-affiliated conference abstract |
| V-024 | EACR 2025 EACR25-1044: SW620 used in 3D spheroid model only — NOT a vaccine component | HIGH | CONFERENCE ABSTRACT (sponsor-affiliated) | EACR25-1044 | Prevents misattribution of SW620 as vaccine cell line | YES |
| V-025 | REGONIVO multiomic: CMS4 subtype associated with IO response (p=0.035); CD8+ TIL + M2 macrophage density higher in responders | MEDIUM | PEER-REVIEWED | PMID 38336371 | Hypothesis-generating biomarker finding; BreAK CRC-001 does not capture CMS subtype | YES — with caveat: retrospective, n=25 |

---

## SECTION 2 — PROVISIONAL CLAIMS REQUIRING CAVEAT TAGS IN 8D OUTPUT

These claims are directionally supported but carry residual uncertainty that 8D must tag explicitly.
8D may use these only with the specified caveat language attached.

| ID | Claim text | Confidence | Source type | Caveat required | Why it matters for 8D | 8D can proceed without sponsor data? |
|----|-----------|-----------|------------|----------------|----------------------|--------------------------------------|
| P-001 | STC-1010 is produced from HCT116, HT29, and LoVo cell lines | MEDIUM | 2× CONFERENCE ABSTRACT (sponsor-affiliated) | Must append: "as reported in two sponsor-affiliated conference abstracts (ASCO 2023 e14661; EACR 2025 EACR25-1044); no independent third-party verification" | Cell line identity is foundational to SC-001 conflict and antigen architecture claims | YES — with caveat |
| P-002 | AACR 2026 CT051: n=6, 0 DLTs, 100% DCR, DTH responses observed in all patients | LOW | PRESS RELEASE ONLY (CheckOrphan summary) | Must append: "press-release summary only; quantitative DTH values, per-patient RECIST assessments, and immune biomarker data not publicly available" | Only available first-in-human data; 8D cannot use quantitative values from this source | YES — but only qualitative framing; no quantitative values |
| P-003 | GRANITE GO-010: all-comer HR 0.73; ctDNA-low subgroup HR 0.50 | LOW | PRESS RELEASE ONLY (company, bankrupt) | Must append: "press-release-only; company bankrupt; ctDNA threshold undefined; wide CIs; not peer-reviewed; do not cite externally as evidence of efficacy" | Strongest available enrichment hypothesis; 8D may use as design hypothesis only | YES — as design hypothesis only, not efficacy evidence |
| P-004 | BOT/BAL Phase 2: cORR 19% (n=62); Phase 3 BATTMAN initiated | MEDIUM | CONFERENCE DATA (ASCO GI 2025) — not peer-reviewed | Must append: "conference presentation only; not yet peer-reviewed" | Confirms Phase 1 signal; use with peer-review caveat | YES — with caveat |
| P-005 | Liver mets + Immunoscore-IC-low = IO non-permissive (nuanced framing) | MEDIUM | PEER-REVIEWED secondary analysis (PMID 39908699) | Must append: "exploratory secondary analysis of Phase 2 trial; pMMR not MSS-restricted; interaction tests non-significant (Pint PFS=0.166, OS=0.473); directional observation only" | Replaces prior overstated canon; 8D must use nuanced framing, not the old binary | YES — with caveat |
| P-006 | pTMB ≥28 mut/Mb (GuardantOMNI) predictive of IO benefit in MSS mCRC | MEDIUM | PEER-REVIEWED (PMID 38727700) | Must append: "data-driven threshold (not pre-specified); Pint=0.070 (borderline); GuardantOMNI-specific; derived from 2L+ refractory population; not validated in 1L; requires prospective validation" | Biomarker hypothesis for Phase IIa enrichment; 8D must not present as validated biomarker | YES — with caveat |
| P-007 | AACR 2024 multi-omics analysis confirmed batch-to-batch reproducibility of STC-1010 | LOW | PRESS RELEASE (BioSpace, company-controlled) | Must append: "press-release only; data package not in public record" | Supports manufacturing consistency claim; 8D cannot use specific data values | YES — qualitative only |
| P-008 | MEDITREME: neoantigen-specific T-cell responses associated with better PFS | MEDIUM | PEER-REVIEWED (PMID 37563240) | Must append: "single-arm Phase 1b/2; 3-month PFS primary endpoint is a weak surrogate; no randomized control" | Supports DTH/T-cell response as pharmacodynamic endpoint; validates BreAK CRC-001 DTH rationale | YES — with caveat |

---

## SECTION 3 — BLOCKED CLAIMS — 8D MUST NOT USE

These claims are either unsupported, contradicted by the verified ledger, or carry source-integrity
failures that make them unsafe for any 8D vector. Using these would introduce errors into the
vector space that cannot be corrected without re-running the full pipeline.

| ID | Claim text | Block reason | Ledger reference | Corrective action before unblocking |
|----|-----------|-------------|-----------------|-------------------------------------|
| B-001 | "Biopsy required" (prior canon) | CONTRADICTED by registry. NCT06934538 Inclusion #6 states conditional biopsy only. | C-007 CONTRADICTED | Replace with V-016: "biopsy is conditional" |
| B-002 | "Liver metastases are the single strongest predictor of IO failure in MSS mCRC" (prior canon, unqualified) | OVERSTATED. AtezoTRIBE 2025 (PMID 39908699) shows liver mets are prognostic but not predictive of IO resistance when Immunoscore-IC is high. The binary framing is no longer defensible. | C-003 NUANCED | Replace with P-005 nuanced framing |
| B-003 | "ctDNA mentioned as exploratory biomarker" in BreAK CRC-001 (prior canon) | NOT-FOUND. ctDNA is not explicitly named in NCT06934538 registry. "Molecular biomarkers" is the registry language. The claim that ctDNA is included is unverified. | C-005 NOT-FOUND | Replace with V-020: "molecular biomarkers listed; ctDNA not explicitly named" |
| B-004 | pTMB ≥28 mut/Mb finding attributed to MEDITREME | TRIAL ATTRIBUTION ERROR. The finding is from CO.26 (PMID 38727700), not MEDITREME. MEDITREME has no pTMB analysis. | AUDIT-04 CORRECTION-01 | Replace with V-013: CO.26 (PMID 38727700) |
| B-005 | GRANITE GO-010 HR values cited without press-release-only tag | SOURCE INTEGRITY FAILURE. These values are press-release-only, company bankrupt, threshold undefined. Citing without explicit tag is a source integrity violation. | C-013 PRESS-RELEASE-ONLY | Attach P-003 caveat before any use |
| B-006 | AACR 2026 CT051 quantitative DTH values (any specific numbers beyond "DTH responses observed") | NOT IN PUBLIC RECORD. No full poster text available. CheckOrphan summary does not contain quantitative DTH data. | C-014 PRESS-RELEASE-ONLY | Blocked until NDA-02 data package received |
| B-007 | "The only MSS CRC IO program with a validated clinical signal" (re: BOT/BAL) | OVERSTATED. RIN NLM ORR 36.4% (PMID 39504677) is also a validated clinical signal in MSS mCRC. BOT/BAL is not the only program. The claim is factually incorrect. | V-009 | Replace with: "BOT/BAL and RIN are the two programs with validated IO signals in no-liver-met MSS mCRC" |
| B-008 | "ORR 36.4% in non-liver-metastatic MSS mCRC is the highest validated IO response rate in this indication" (re: RIN) | PARTIALLY OVERSTATED. RIN NLM n=22 is a small subgroup. The claim is directionally correct but "highest validated" overstates the certainty given n=22. | C-012 MEDIUM confidence | Replace with: "ORR 36.4% in the RIN NLM subgroup (n=22) is among the highest reported IO response rates in MSS mCRC" |
| B-009 | "Six-layer immunosuppressive architecture of MSS CRC — TGF-β-driven exclusion, VEGF-mediated myeloid suppression, MHC-I downregulation, Treg enrichment, M2 macrophage polarization, and liver-mediated immune tolerance" | INFERRED SYNTHESIS. This six-layer framing is a mechanistic synthesis not directly sourced to a single peer-reviewed publication. It is a reasonable scientific summary but is not a citable claim. | NOT IN VERIFIED LEDGER | Tag as "mechanistic synthesis" or source each layer to a specific publication before 8D use |
| B-010 | "This is a design failure, not a biology failure" (re: GRANITE GO-010) | INTERPRETIVE CLAIM. The distinction between design failure and biology failure for GRANITE GO-010 is an editorial interpretation, not a finding from the data. The trial failed; the reason is not established from public sources. | NOT IN VERIFIED LEDGER | Remove or tag as "editorial interpretation" |
| B-011 | "Bevacizumab has well-characterized immunomodulatory effects — VEGF blockade reduces immunosuppressive myeloid cell recruitment, normalizes tumor vasculature to improve T-cell trafficking" | INFERRED MECHANISM. The immunomodulatory effects of bevacizumab are biologically plausible and supported by preclinical literature, but the specific claim that this confounds BreAK CRC-001 immune biomarker data is an inference, not a finding from the trial. | NOT IN VERIFIED LEDGER | Tag as "biologically plausible mechanism; not directly tested in BreAK CRC-001" |
| B-012 | "The AACR 2024 press release mentions multi-omics analysis of STC-1010 batches confirming batch-to-batch reproducibility — this data package presumably includes antigen characterization per cell line" | SPECULATION. "Presumably includes" is not a supported claim. The press release confirms multi-omics analysis exists; it does not confirm antigen characterization per cell line is included. | NOT IN VERIFIED LEDGER | Remove "presumably includes antigen characterization per cell line" — replace with "the data package is not in the public record" |

---

## SECTION 4 — CONTRADICTIONS AND DRIFT BETWEEN CURRENT DRAFTS AND VERIFIED LEDGER

These are specific sentences or passages in the Design Intelligence Brief (brenus_outreach_brief.md)
that are inconsistent with the verified ledger. Presented as a redline-style adjudication table.
Format: Brief location | Brief text (verbatim) | Ledger verdict | Disposition

| # | Brief location | Brief text (verbatim or paraphrased) | Ledger verdict | Disposition |
|---|---------------|--------------------------------------|---------------|-------------|
| D-001 | §2.2, BOT/BAL paragraph | "The only MSS CRC IO program with a validated clinical signal." | CONTRADICTED — RIN NLM ORR 36.4% (PMID 39504677) is also a validated signal. | REPLACE with: "One of two programs with validated IO signals in no-liver-met MSS mCRC (BOT/BAL and RIN)." |
| D-002 | §2.2, RIN paragraph | "ORR 36.4% in non-liver-metastatic MSS mCRC is the highest validated IO response rate in this indication." | OVERSTATED — n=22 NLM subgroup; "highest validated" overstates certainty. | REPLACE with: "ORR 36.4% in the RIN NLM subgroup (n=22) is among the highest reported IO response rates in MSS mCRC." |
| D-003 | §2.3, GRANITE paragraph | "This is a design failure, not a biology failure." | NOT IN VERIFIED LEDGER — interpretive claim not supported by data. | REMOVE or tag as "editorial interpretation." |
| D-004 | §2.1, CheckMate 9X8 lesson | "PD-1 blockade alone cannot overcome the six-layer immunosuppressive architecture of MSS CRC — TGF-β-driven exclusion, VEGF-mediated myeloid suppression, MHC-I downregulation, Treg enrichment, M2 macrophage polarization, and liver-mediated immune tolerance." | NOT IN VERIFIED LEDGER — mechanistic synthesis not sourced to a single peer-reviewed publication. | TAG as "mechanistic synthesis" or source each layer individually. |
| D-005 | §3, Constraint 3 | "Bevacizumab has well-characterized immunomodulatory effects — VEGF blockade reduces immunosuppressive myeloid cell recruitment, normalizes tumor vasculature to improve T-cell trafficking, and modulates the tumor immune microenvironment." | NOT IN VERIFIED LEDGER — biologically plausible but not sourced in the ledger. | TAG as "biologically plausible mechanism; not directly tested in BreAK CRC-001" or add specific citations. |
| D-006 | §4, antigen architecture | "The AACR 2024 press release mentions multi-omics analysis of STC-1010 batches confirming batch-to-batch reproducibility — this data package presumably includes antigen characterization per cell line." | SPECULATION — "presumably includes" is not supported. | REMOVE "presumably includes antigen characterization per cell line." Replace with: "the data package is not in the public record." |
| D-007 | §5, Question 1 | "The AACR 2026 CT051 press release reports DTH responses in all 6 Phase Ia patients — this is the right readout." | PARTIALLY CORRECT but source is press-release-only (CheckOrphan summary). The brief does not label this as press-release-only in this passage. | ADD source label: "per press-release summary (CheckOrphan); quantitative DTH data not publicly available." |
| D-008 | §6, Enrichment hypothesis testing | "Using the existing Phase Ia data (n=6, DTH responses, 100% DCR) to generate a prospective enrichment hypothesis for Phase IIa." | PARTIALLY CORRECT — n=6 and 100% DCR are from press-release-only source (C-014). The brief presents these as usable data without the press-release caveat in this passage. | ADD: "based on press-release summary only; quantitative data not publicly available." |
| D-009 | §6, Translational data interpretation | "The EACR 2025 3D spheroid data (HCT116, HT29, SW620, LoVo co-aggregated with CCD-18Co fibroblasts; CD8+ T-cell killing; IFN-γ production)" | PARTIALLY CORRECT — IFN-γ production is from ASCO 2023 e14661 (CAM model), not EACR 2025 spheroid data. EACR 2025 reports CD8+ T-cell killing and cytokine secretion but does not specifically name IFN-γ in the spheroid model. | VERIFY: IFN-γ attribution. If from ASCO 2023 CAM model, label accordingly. Do not conflate CAM and spheroid model readouts. |
| D-010 | §8, Evidence base | "CONFERENCE ABSTRACT: ASCO 2023 e14661 (STC-1010 cell lines, Alzeeb et al.), EACR 2025 EACR25-1044 (STC-1010 3D spheroids, Zbinden et al.)" | CORRECT but incomplete — closure audit established that both are sponsor-affiliated. The source integrity note should specify this. | ADD: "both are sponsor-affiliated; no independent third-party verification of cell line identity." |
| D-011 | §2.3, Architecture failures | "Sintilimab + bev + XELOX (NCT04194359): ORR 84%, n=25, single-arm, single-center, Chinese population. Uninterpretable." | CORRECT — but the brief does not note that this is a peer-reviewed publication (PMID 37554125). The "uninterpretable" label is correct but should be sourced. | ADD PMID 37554125 as source. |
| D-012 | §3, Constraint 4 | "A secondary analysis of CO.26 (Loree et al., Clin Cancer Res 2024, PMID 38727700) found that plasma TMB ≥28 mut/Mb (GuardantOMNI) was predictive of benefit from durvalumab + tremelimumab in refractory MSS mCRC (HR 0.34, Pint=0.070)." | CORRECT — trial attribution is correct (CO.26, not MEDITREME). This is the corrected version. | ACCEPT — no change needed. |

---

## SECTION 5 — MISSING SPONSOR DATA REQUIRED BEFORE FULL VECTOR ASSIGNMENT

These are data gaps that cannot be resolved from public sources. 8D vector assignment for the
affected dimensions is blocked or degraded until sponsor data is received.

| ID | Missing data | Affected 8D dimensions | Current 8D status | Sponsor data request | Can 8D proceed with degraded vector? |
|----|-------------|----------------------|------------------|---------------------|--------------------------------------|
| M-001 | CT051 full poster: quantitative DTH values, per-patient RECIST, immune biomarker data, translational correlation (n=6) | Pharmacodynamic signal strength; immune engagement vector; clinical proof-of-concept | DEGRADED — only qualitative "DTH responses observed" available | NDA-02 (Priority 1) | YES — but pharmacodynamic vector will be LOW confidence; no quantitative values |
| M-002 | Antigen overlap data: HT29-derived antigens (BRAF V600E) vs HCT116/LoVo antigens (BRAF WT) vs patient tumor proteomes | Antigen mismatch risk vector; SC-001 conflict resolution | BLOCKED — SC-001 conflict is confirmed; clinical consequence unknown | NDA-01 (Priority 1) | NO — antigen architecture vector cannot be assigned without this data |
| M-003 | Liver met stratification plan: is liver-met status captured as stratification variable or pre-specified covariate in Phase IIa? | Patient selection vector; Phase IIa interpretability score | DEGRADED — registry confirms gap; sponsor intent unknown | NDA-03 (Priority 1) | YES — but patient selection vector will be flagged as UNRESOLVED |
| M-004 | ctDNA platform and threshold specification: is ctDNA included in molecular biomarker panel? What platform? What threshold? | Molecular biomarker vector; enrichment hypothesis testability | BLOCKED — ctDNA not explicitly named in registry | NDA-04 (Priority 2) | NO — molecular biomarker vector cannot be assigned without platform and threshold |
| M-005 | Bevacizumab stratification plan: is bev use captured as stratification variable or pre-specified covariate? | Backbone heterogeneity vector; immune biomarker confounding risk | DEGRADED — registry confirms optional bev; sponsor intent unknown | NDA-05 (Priority 2) | YES — but backbone heterogeneity vector will be flagged as HIGH CONFOUNDING RISK |
| M-006 | KRAS subtype (G12C, G12D, G12V, G13D) and SMAD4 status capture in eCRF | Molecular subgroup vector; liver-met tropism risk | DEGRADED — KRAS mutation required but subtype not specified in registry | NDA-06 (Priority 2) | YES — but molecular subgroup vector will be LOW resolution |
| M-007 | Biopsy compliance rate from Phase Ia experience; biomarker panel specification (IHC, RNA-seq, proteomics) | Tissue biomarker vector; Immunoscore-IC and CMS subtype capture | DEGRADED — conditional biopsy confirmed; compliance rate unknown | NDA-07 (Priority 2) | YES — but tissue biomarker vector will be flagged as MISSING DATA RISK |
| M-008 | Full multi-omics antigen characterization package: proteomics per cell line, batch reproducibility data | Antigen breadth vector; manufacturing consistency | DEGRADED — press release confirms data exists internally; not in public record | NDA-10 (Priority 3) | YES — but antigen breadth vector will be LOW confidence |

---

## SUMMARY — 8D READINESS SCORECARD

| Category | Count | 8D status |
|----------|-------|-----------|
| VERIFIED — safe for 8D use | 25 claims | PROCEED |
| PROVISIONAL — require caveat tags | 8 claims | PROCEED WITH TAGS |
| BLOCKED — 8D must not use | 12 claims | DO NOT USE |
| CONTRADICTIONS/DRIFT in brief | 12 items | REPAIR BEFORE USE |
| MISSING SPONSOR DATA (vector-blocking) | 2 dimensions (M-002, M-004) | BLOCKED |
| MISSING SPONSOR DATA (vector-degraded) | 6 dimensions (M-001, M-003, M-005, M-006, M-007, M-008) | DEGRADED |

**Overall 8D readiness:** PARTIAL. The comparator failure landscape (V-001 through V-015) and
the BreAK CRC-001 registry gaps (V-016 through V-022) are clean and ready for vector assignment.
The antigen architecture vector (M-002) and molecular biomarker vector (M-004) are blocked
pending sponsor data. The first-in-human pharmacodynamic vector (M-001) is degraded to
qualitative-only. Twelve items in the Design Intelligence Brief require repair before the brief
can be used as a 8D input document.
"""

import os
os.makedirs("/workspace/phd_outputs", exist_ok=True)
with open("/workspace/phd_outputs/8d_readiness_audit.md", "w") as f:
    f.write(audit_8d)

print("Saved 8d_readiness_audit.md")
print(f"Length: {len(audit_8d):,} characters")
print("\nSummary:")
print("  VERIFIED claims safe for 8D: 25")
print("  PROVISIONAL (caveat tags required): 8")
print("  BLOCKED (8D must not use): 12")
print("  CONTRADICTIONS/DRIFT in brief: 12")
print("  MISSING SPONSOR DATA — vector-blocking: 2 (M-002 antigen overlap, M-004 ctDNA)")
print("  MISSING SPONSOR DATA — vector-degraded: 6")


Saved 8d_readiness_audit.md
Length: 25,800 characters

Summary:
  VERIFIED claims safe for 8D: 25
  PROVISIONAL (caveat tags required): 8
  BLOCKED (8D must not use): 12
  CONTRADICTIONS/DRIFT in brief: 12
  MISSING SPONSOR DATA — vector-blocking: 2 (M-002 antigen overlap, M-004 ctDNA)
  MISSING SPONSOR DATA — vector-degraded: 6
